# ANALÍTICA EMPRESARIAL INTEGRADA
## Laboratorio 10 · Pronóstico de demanda y decisiones de abastecimiento

**Estudiante:** Christopher Rojas Zuñiga  
**Carrera:** Big Data y Ciencia de Datos · TECSUP  
**Periodo:** 2026-II · Semana 10  
**Docente:** Pilar Rocío Sayán Mejía  
**Fuente de datos:** Online Retail II (UCI Machine Learning Repository, ID 502)

### Enfoque del trabajo
En este laboratorio estudio las ventas de una empresa de comercio electrónico y comparo modelos de pronóstico semanal.
Mi interés es conectar los resultados de analítica con decisiones de inventario: no basta con predecir una cifra;
es necesario calcular cuánto cuesta equivocarse, comprobar la estabilidad del modelo y definir si puede usarse en operaciones.

**Aporte adicional de esta versión:** incorporo indicadores de servicio, un tablero comparativo y un ejercicio de compras netas con inventario simulado, pensando en cómo aplicaría el enfoque al abastecimiento de repuestos y equipos industriales. **Los productos del dataset son artículos minoristas del Reino Unido, no registros de una empresa de maquinaria.**

> **Estado del archivo:** adaptación técnica preparada para ejecutarse y verificarse en Google Colab. Las cifras explicativas del material de partida deben contrastarse con las tablas obtenidas al volver a ejecutar todas las celdas antes de entregar.


## Datos del estudiante y entrega

| Campo | Registro |
|---|---|
| Nombre | **Christopher Rojas Zuñiga** |
| Carrera | Big Data y Ciencia de Datos — TECSUP |
| Sección | **Completar con mi sección matriculada** |
| Repositorio personal | `https://github.com/christopherrojasz-dot/analitica-empresarial-integrada` |
| Carpeta solicitada para este archivo | `seman 10/` |
| Nombre del notebook | `LAB_10_AEI_CHRISTOPHER_ROJAS_ZUNIGA.ipynb` |
| Enlace directo en GitHub | **Pendiente de publicación** |
| Enlace a Colab | **Pendiente de guardar y compartir desde mi cuenta** |

**Proceso de entrega.** Primero ejecutar **Entorno de ejecución → Ejecutar todas**, revisar tablas, gráficos y
conclusiones, y guardar una copia en Google Drive. Luego descargar el `.ipynb` e invocar el bloque de publicación
que aparece al final. GitHub crea la carpeta al publicar un archivo dentro de ella.

**Nota sobre la ruta:** la indicación incluida en el laboratorio compartido menciona `semana10/`, mientras que la
carpeta aquí solicitada es literalmente **`seman 10/`**. Si la plataforma de evaluación exige el nombre de la consigna,
cambiar `CARPETA_DESTINO` a `semana10` antes de publicar.

**Trazabilidad:** se empleó un notebook de laboratorio proporcionado como referencia de la actividad. Esta
versión incluye reorganización, explicaciones personales y ampliaciones de análisis con asistencia de IA.
Los resultados deben reproducirse y comprobarse en la propia ejecución antes de presentarse como evidencia.


# 1. Problema empresarial y estrategia de análisis

**Decisión que busco respaldar:** ¿cuántas unidades se deberían tener disponibles cada semana y qué alternativa
minimiza el costo de quedarse corto o terminar con exceso de inventario?

Para resolverlo separo **cuatro decisiones** que en una empresa no deberían confundirse:

1. **Pronosticar:** estimar unidades vendidas, a partir del historial.
2. **Dimensionar:** decidir una cantidad objetivo incorporando o no margen de seguridad.
3. **Costear:** poner en dinero los sobrantes y faltantes según supuestos documentados.
4. **Aprobar:** decidir si un modelo merece un piloto o si conviene conservar el método sencillo.

| Elemento | Diseño del experimento |
|---|---|
| Periodicidad | Semanas completas de lunes a domingo |
| Variable | Unidades de ventas registradas por SKU en Reino Unido |
| Test final | 8 últimas semanas completas, reservado hasta elegir modelo y política |
| Selección de producto | Solo información anterior al test |
| Validación | Backtesting rolling-origin, horizontes de 8 semanas |
| Modelos | Naïve, Naïve estacional (m = 4) y Exponential Smoothing |
| Elección | Menor costo medio, con tolerancia del 1 % y desempates previamente definidos |
| Protección | Colchón 0, 0,5, 1 o 1,5 multiplicado por MAE previo, sin mirar el futuro |

**Criterio profesional:** un error grande no siempre produce el mayor daño operativo. En mantenimiento o
comercialización de maquinaria, por ejemplo, faltar un componente crítico puede detener una operación, mientras
que otro repuesto de baja rotación podría inmovilizar capital por meses. En este dataset aplicaré **los costos
académicos en libras**, no costos reales de la industria peruana.


## Actividad 1 · Conceptos con ejemplos prácticos

Explico cada término desde la perspectiva de quien debe transformar datos históricos en decisiones operativas.

| Concepto | Cómo lo entiendo y ejemplo | Idea central |
|---|---|---|
| Serie temporal | Son observaciones registradas con fecha y orden. Ejemplo: unidades despachadas por semana de un mismo producto. Si mezclo las fechas ya no conozco qué pasó antes de cada pronóstico. | El orden sí importa. |
| Semana completa | Uso semanas con cobertura del lunes al domingo para comparar periodos equivalentes. Si la primera empezó martes, tendría un día menos registrado. | Evitar semanas parcialmente observadas. |
| Tendencia y estacionalidad | La tendencia describe un cambio sostenido del nivel; la estacionalidad es una repetición consistente. Un pico de campaña no demuestra por sí solo un ciclo. | No confundir ruido con patrón. |
| Baseline | Es mi punto de comparación. Naïve repite la última semana; el estacional repite lo ocurrido hace cuatro semanas. | Un modelo avanzado debe justificar su costo adicional. |
| Suavizamiento exponencial | Resume el historial dando pesos distintos a las observaciones; puede incluir nivel, tendencia y ciclo. | Más parámetros no significa mejor decisión. |
| Partición temporal | Primero entreno con el pasado, luego comparo en validación y reservo el test. | No usar información futura para seleccionar. |
| Rolling-origin | Repito la prueba partiendo de varias fechas históricas, siempre prediciendo las semanas posteriores. | Analizar estabilidad, no solo una fecha favorable. |
| MAE, RMSE y WAPE | MAE expresa error promedio en unidades; RMSE penaliza más un fallo grande; WAPE relativiza el error absoluto al volumen. | Cada métrica responde una pregunta distinta. |
| Costo asimétrico | Si faltar cuesta £1,50 y sobrar £0,40, un faltante de 20 unidades cuesta £30 y un excedente igual cuesta £8. | Importa el signo del error. |
| Pronóstico vs. compra | La cifra pronosticada no es la orden de compra. Primero debo considerar stock disponible, reposiciones confirmadas y tiempos de entrega. | Pronosticar ≠ comprar. |


In [1]:
%pip install -q ucimlrepo statsmodels plotly scikit-learn openpyxl

In [2]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sklearn
import statsmodels

from ucimlrepo import fetch_ucirepo
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

# No se ocultan advertencias globalmente: las de cada ajuste se registran y se revisan al final.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
             "statsmodels": statsmodels.__version__, "scikit-learn": sklearn.__version__}
print("Versiones:", VERSIONES)

Versiones: {'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


# 2. Descarga de datos reales y trazabilidad

Se intenta primero la API oficial de UCI; si falla, el archivo ZIP oficial, del que se leen y concatenan sus **dos hojas**; si ambas fallan, se sube manualmente el archivo descargado desde https://archive.ics.uci.edu/dataset/502. La celda registra fuente, hojas, dimensiones, rango de fechas y versiones.

> Que otra copia tenga el mismo número de filas **no demuestra** que sea idéntica. Con otras versiones de las bibliotecas, los ajustes de Exponential Smoothing pueden producir cifras ligeramente distintas.

In [3]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

def leer_dos_hojas(fuente_excel):
    hojas = pd.read_excel(fuente_excel, sheet_name=None)  # "Year 2009-2010" y "Year 2010-2011"
    return pd.concat(hojas.values(), ignore_index=True), list(hojas)

df, fuente, hojas_leidas = None, None, None
try:
    retail = fetch_ucirepo(id=502)
    df, fuente, hojas_leidas = retail.data.original.copy(), "API ucimlrepo (UCI ID 502)", "tabla única entregada por la API"
except Exception as exc:
    print("API ucimlrepo no disponible:", type(exc).__name__)

if df is None:
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as respuesta:
            blob = respuesta.read()
        with ZipFile(BytesIO(blob)) as z:
            nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
            df, hojas_leidas = leer_dos_hojas(BytesIO(z.read(nombre)))
        fuente = "ZIP oficial de UCI"
    except Exception as exc:
        print("ZIP oficial no disponible:", type(exc).__name__)

if df is None:
    locales = sorted(f for f in os.listdir(".")
                     if "online" in f.lower() and "retail" in f.lower() and f.lower().endswith((".xlsx", ".csv")))
    if not locales:
        try:
            from google.colab import files
            print("Suba el archivo online_retail_II.xlsx descargado desde UCI.")
            locales = list(files.upload().keys())
        except ImportError:
            raise FileNotFoundError("Coloque online_retail_II.xlsx en la carpeta de trabajo y vuelva a ejecutar.")
    archivo = locales[0]
    if archivo.lower().endswith(".xlsx"):
        df, hojas_leidas = leer_dos_hojas(archivo)
    else:
        df = pd.read_csv(archivo, dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str})
        hojas_leidas = "no aplica (CSV)"
    fuente = f"archivo local: {archivo}"

fechas_raw = pd.to_datetime(df[[c for c in df.columns if "date" in str(c).lower()][0]], errors="coerce")
print("Fuente utilizada:", fuente)
print("Hojas:", hojas_leidas)
print("Dimensión:", df.shape)
print("Rango de fechas:", fechas_raw.min(), "→", fechas_raw.max())
print("Ejecución:", pd.Timestamp.now(tz="UTC").isoformat())
print("Versiones:", VERSIONES)
display(df.head())

API ucimlrepo no disponible: DatasetNotFoundError
Fuente utilizada: ZIP oficial de UCI
Hojas: ['Year 2009-2010', 'Year 2010-2011']
Dimensión: (1067371, 8)
Rango de fechas: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Ejecución: 2026-10-28T02:52:43.568558+00:00
Versiones: {'python': '3.13.18', 'pandas': '2.2.3', 'numpy': '2.1.3', 'statsmodels': '0.15.0', 'scikit-learn': '1.6.1'}


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_", ""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_", "")
        if key in normalized:
            return normalized[key]
    return None

mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "CustomerID": find_col(df.columns, ["Customer ID", "CustomerID"]),
    "Country": find_col(df.columns, ["Country"]),
}
faltan = [k for k, v in mapping.items() if v is None and k not in ("Description", "CustomerID")]
if faltan:
    raise ValueError(f"No se identificaron columnas obligatorias: {faltan}")
df = df.rename(columns={v: k for k, v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID', 'Country': 'Country'}


# 3. Control de calidad

Los `StockCode` nulos se descartan **antes** de convertir a texto; de lo contrario, la ausencia se convertiría en un producto llamado «nan».

In [5]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
print("StockCode nulos descartados:", int(df["StockCode"].isna().sum()))
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df = df[df["StockCode"] != ""]
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

print("Fecha mínima:", df["InvoiceDate"].min(), "·", df["InvoiceDate"].min().day_name())
print("Fecha máxima:", df["InvoiceDate"].max(), "·", df["InvoiceDate"].max().day_name())
print("Cancelaciones:", int(cancel.sum()))
print("Quantity <= 0:", int((df["Quantity"] <= 0).sum()))
print("UnitPrice <= 0:", int((df["UnitPrice"] <= 0).sum()))
print("Fechas nulas:", int(df["InvoiceDate"].isna().sum()))

clean = df.loc[(~cancel) & df["InvoiceDate"].notna() & (df["Quantity"] > 0) &
               (df["UnitPrice"] > 0) & df["Country"].notna()].copy()
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
print("Filas válidas UK:", len(uk), "· SKUs UK:", uk["StockCode"].nunique())

StockCode nulos descartados: 0
Fecha mínima: 2009-12-01 07:45:00 · Tuesday
Fecha máxima: 2011-12-09 12:50:00 · Friday
Cancelaciones: 19494
Quantity <= 0: 22950
UnitPrice <= 0: 6207
Fechas nulas: 0
Filas válidas UK: 958501 · SKUs UK: 4905


### Pregunta 1 · ¿Por qué depurar las cantidades?

**Mi respuesta:** No puedo sumar directamente `Quantity` y asumir que todo son ventas. En el archivo hay
cancelaciones, devoluciones y ajustes; si los mezclo con los despachos, la serie quedaría artificialmente baja
y el modelo aprendería movimientos contables como si fueran cambios de demanda. El notebook separa estas
operaciones y exige cantidad, precio y fecha válidos. Esto se parece a analizar repuestos: una devolución de
almacén no equivale a una venta negativa del mismo día.


# 4. Semanas completas y partición temporal

Las semanas van de **lunes a domingo** y se identifican por la fecha del lunes. Una semana es **completa** solo si el archivo cubre sus siete días: se excluyen la primera semana si el archivo no empieza en lunes y la última si no termina en domingo. Las **últimas 8 semanas completas** se reservan como test final.

Algunas semanas no tienen **ninguna venta registrada** en los datos filtrados. El cierre de la tienda es una **posible explicación no verificada**: los datos no lo confirman.

In [6]:
def lunes(fechas):
    fechas = pd.to_datetime(fechas)
    return fechas.dt.normalize() - pd.to_timedelta(fechas.dt.weekday, unit="D")

uk["week"] = lunes(uk["InvoiceDate"])
inicio_archivo, fin_archivo = df["InvoiceDate"].min(), df["InvoiceDate"].max()
lunes_inicio = inicio_archivo.normalize() - pd.Timedelta(days=inicio_archivo.weekday())
lunes_fin = fin_archivo.normalize() - pd.Timedelta(days=fin_archivo.weekday())
primera_completa = lunes_inicio if inicio_archivo.weekday() == 0 else lunes_inicio + pd.Timedelta(weeks=1)
ultima_completa = lunes_fin if fin_archivo.weekday() == 6 else lunes_fin - pd.Timedelta(weeks=1)

semanas = pd.date_range(primera_completa, ultima_completa, freq="W-MON")
TEST_H = 8
semanas_test = semanas[-TEST_H:]
semanas_train = semanas[:-TEST_H]

registros = uk.groupby("week").size().reindex(semanas, fill_value=0)
semanas_sin_registro = registros[registros == 0].index

print("Semanas excluidas por incompletas:", lunes_inicio.date(), "y", lunes_fin.date())
print("Semanas completas:", len(semanas), "·", semanas[0].date(), "→", semanas[-1].date())
print("Entrenamiento:", len(semanas_train), "semanas ·", semanas_train[0].date(), "→", semanas_train[-1].date())
print("Test final:", len(semanas_test), "semanas ·", semanas_test[0].date(), "→", semanas_test[-1].date())
print("Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada):",
      [d.date() for d in semanas_sin_registro])

Semanas excluidas por incompletas: 2009-11-30 y 2011-12-05
Semanas completas: 104 · 2009-12-07 → 2011-11-28
Entrenamiento: 96 semanas · 2009-12-07 → 2011-10-03
Test final: 8 semanas · 2011-10-10 → 2011-11-28
Semanas sin ventas registradas en los datos filtrados (cierre: explicación posible no verificada): [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]


### Pregunta 2 · ¿Por qué no dividir los datos al azar?

**Mi respuesta:** Una división aleatoria permitiría entrenar con ventas de noviembre para pronosticar octubre.
Eso no ocurre al decidir compras reales: hoy solo dispongo del historial hasta hoy. Mantengo la secuencia
cronológica, hago la selección en entrenamiento/validación y dejo las últimas ocho semanas intactas hasta la
comparación final. Así el error que observo se aproxima mejor al que tendría al operar.


# 5. Selección reproducible del producto (solo con el entrenamiento)

Criterios, calculados **exclusivamente con las semanas de entrenamiento**:
1. **Periodo con ventas registradas cercano al entrenamiento completo:** primera venta dentro de las **4 primeras** semanas y última venta dentro de las **4 últimas**. El criterio tolera hasta 3 semanas iniciales o finales sin ventas, que luego se completan con cero aunque podrían corresponder a un periodo sin disponibilidad. **No verifica inventario.**
2. **Cobertura:** ventas registradas en al menos el 90 % de las semanas de entrenamiento con algún registro en la tienda.
3. Entre los elegibles, el de **mayor cantidad de unidades**; desempate por más semanas con ventas.

Se comprueba que haya al menos un producto elegible (y al menos dos para el Reto 3).

In [7]:
uk_train = uk[uk["week"].isin(semanas_train)]
semanas_con_registro_train = [w for w in semanas_train if w not in set(semanas_sin_registro)]

cobertura = (uk_train.groupby(["StockCode", "week"])["Quantity"].sum().reset_index()
             .groupby("StockCode")
             .agg(semanas_con_venta=("week", "nunique"), unidades=("Quantity", "sum"),
                  primera_venta=("week", "min"), ultima_venta=("week", "max"))
             .reset_index())
cobertura["cobertura"] = cobertura["semanas_con_venta"] / len(semanas_con_registro_train)
elegibles = cobertura[
    (cobertura["primera_venta"] <= semanas_train[3]) &      # dentro de las 4 primeras semanas
    (cobertura["ultima_venta"] >= semanas_train[-4]) &      # dentro de las 4 últimas semanas
    (cobertura["cobertura"] >= 0.90)
].sort_values(["unidades", "semanas_con_venta"], ascending=False).reset_index(drop=True)

if len(elegibles) == 0:
    raise ValueError("Ningún SKU cumple los criterios: revise la carga y la limpieza antes de continuar.")
print("SKUs elegibles:", len(elegibles), "· suficientes para el Reto 3:", len(elegibles) >= 2)
SKU = elegibles.loc[0, "StockCode"]
desc = uk.loc[uk["StockCode"] == SKU, "Description"].dropna().astype(str).mode().iloc[0]
print("SKU seleccionado:", SKU, "·", desc)
display(elegibles.head(5))

SKUs elegibles: 389 · suficientes para el Reto 3: True
SKU seleccionado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS


,StockCode,semanas_con_venta,unidades,primera_venta,ultima_venta,cobertura
0,84077,93,88273,2009-12-07,2011-10-03,0.989362
1,85123A,94,80903,2009-12-07,2011-10-03,1.000000
2,85099B,94,79134,2009-12-07,2011-10-03,1.000000
3,21212,94,67878,2009-12-07,2011-10-03,1.000000
4,17003,88,67514,2009-12-07,2011-10-03,0.936170


# 6. Serie semanal de ventas registradas

**Qué representa un cero.** Un cero indica que **no hay ventas registradas** del SKU en esa semana dentro de los datos filtrados. Los datos no permiten distinguir entre:
- **disponibilidad de inventario** con demanda nula;
- **demanda no satisfecha** por falta de inventario (no queda registrada);
- **ausencia de registros** (por ejemplo, semanas sin ninguna venta registrada en la tienda).

Completar con cero es un **supuesto sobre las ventas observadas**: no demuestra demanda cero ni disponibilidad continua. Por eso el laboratorio pronostica **ventas registradas**, que solo aproximan la demanda.

In [8]:
serie = (uk[uk["StockCode"] == SKU].groupby("week")["Quantity"].sum()
         .reindex(semanas, fill_value=0).astype(float))
serie.name = "unidades"
train = serie.loc[semanas_train]
test = serie.loc[semanas_test]

ceros = serie[serie == 0].index
print("Semanas de la serie:", len(serie), "· semanas sin ventas registradas del SKU:", len(ceros))
print("  coinciden con semanas sin ningún registro en la tienda:", [d.date() for d in ceros if d in set(semanas_sin_registro)])
print("  semanas con registros en la tienda pero sin ventas del SKU:", [d.date() for d in ceros if d not in set(semanas_sin_registro)])
print("Unidades en entrenamiento:", int(train.sum()))

Semanas de la serie: 104 · semanas sin ventas registradas del SKU: 3
  coinciden con semanas sin ningún registro en la tienda: [datetime.date(2009, 12, 28), datetime.date(2010, 12, 27)]
  semanas con registros en la tienda pero sin ventas del SKU: [datetime.date(2011, 1, 17)]
Unidades en entrenamiento: 88273


# 7. Exploración (solo con el entrenamiento)

La exploración que orienta decisiones usa únicamente el entrenamiento: mirar el test antes de elegir el modelo sería una fuga de información. En este laboratorio las medias móviles de las **últimas 4 y 8 semanas** se usan para **explorar** el nivel; no se evalúan como método de pronóstico (podrían usarse como tal, pero no forman parte de esta comparación).

In [9]:
explora = pd.DataFrame({"ventas": train})
explora["media_movil_4"] = explora["ventas"].rolling(4).mean()
explora["media_movil_8"] = explora["ventas"].rolling(8).mean()
fig = go.Figure()
for col, nombre in [("ventas", "Ventas registradas"), ("media_movil_4", "Media móvil 4"), ("media_movil_8", "Media móvil 8")]:
    fig.add_trace(go.Scatter(x=explora.index, y=explora[col], name=nombre))
fig.update_layout(title=f"Entrenamiento · SKU {SKU} · {desc}", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

perfil = (explora.assign(semana_anio=explora.index.isocalendar().week.astype(int))
          .groupby("semana_anio")["ventas"].mean().reset_index())
fig = px.line(perfil, x="semana_anio", y="ventas", markers=True,
              title="Perfil promedio por semana del año (entrenamiento)",
              labels={"semana_anio": "Semana del año", "ventas": "Ventas promedio"})
fig.show()

anual = train.groupby(train.index.year).agg(["count", "mean", "median", "max"])
display(anual.round(1))

,count,mean,median,max
2009,4,493.0,384.0,1204.0
2010,52,1034.4,608.0,7063.0
2011,40,812.8,558.0,3216.0


### Pregunta 3 · ¿Veo tendencia, patrones o semanas atípicas?

**Mi análisis del entrenamiento:** Yo no afirmaría que existe una tendencia descendente con solo comparar
los promedios de 2010 (1 034,4 unidades por semana) y 2011 (812,8): el segundo año del entrenamiento se corta
en octubre. Esa diferencia de cobertura puede ocultar la temporada alta y sesgar la conclusión.

Sí se observan **ventas extraordinarias**: en octubre y noviembre de 2010 aparecen semanas de 3 504, 5 301
y 7 063 unidades, muy por encima de la mediana general (584,5). Esto sugiere una campaña o una temporada
fuerte, pero un único otoño completo no basta para validar una estacionalidad anual. La periodicidad de cuatro
semanas que luego pruebo es una hipótesis de modelado, no un hecho confirmado.

También encuentro picos en marzo y agosto de 2010 y marzo de 2011, que podrían provenir de pedidos
mayoristas. Los tres registros semanales con valor cero exigen cuidado: dos coinciden con semanas sin registros
de la tienda y otro corresponde solo al SKU. **Cero registrado no demuestra demanda cero**, del mismo modo
que no vender una pieza una semana no prueba que nadie la solicitó si pudo faltar en almacén.

**Mi siguiente comprobación** sería cruzar picos con facturas y disponibilidad de inventario, en vez de
atribuir automáticamente todo a estacionalidad.


# 8. Modelos, cantidad operativa, costo y regla de selección

- **Naïve:** repite la última semana observada.
- **Naïve estacional (m = 4):** repite la semana de hace 4 semanas. El periodo de 4 semanas es un **supuesto de modelado** (ciclo operativo corto), no una propiedad comprobada del negocio; con 96 semanas de entrenamiento no hay suficientes años para estimar con confianza un ciclo anual de 52 semanas.
- **Exponential Smoothing:** nivel y tendencia aditivos con estacionalidad aditiva m = 4. Si un entrenamiento tuviera menos de 2 × 4 + 8 = 16 semanas, se simplifica a nivel y tendencia (documentado; se informa si ocurre).

**Cantidad operativa:** todo pronóstico se convierte en `max(redondeo(pronóstico), 0)` y **todas** las métricas y costos se calculan sobre esa cantidad.

**Costo base (supuesto académico, no proviene de UCI):** £0,40 por unidad excedente y £1,50 por unidad faltante.

**Regla única de selección** (se usa para el modelo, la política de colchón, el Reto 1 y el Reto 3):
1. Candidatos: costo promedio de validación ≤ mínimo × 1,01 (**incluye** exactamente el 1 %; se admite una holgura de 1e-9 por redondeo numérico). Si el mínimo es 0, solo son candidatos los costos iguales a 0.
2. Desempate entre candidatos: menor MAE promedio, luego menor desviación del costo y, al final, menor complejidad.

In [10]:
SEASON = 4
OVER_COST, SHORT_COST = 0.40, 1.50
MODELOS = ["Naive", "SeasonalNaive", "ExpSmoothing"]   # en orden de complejidad
TOLERANCIA = 0.01
registro_ajustes = []

def ajustar_es(y, etiqueta):
    """Ajusta Exponential Smoothing y registra todas las advertencias del ajuste."""
    if len(y) >= 2 * SEASON + 8:
        modelo = ExponentialSmoothing(y, trend="add", seasonal="add", seasonal_periods=SEASON,
                                      initialization_method="estimated")
        especificacion = "nivel + tendencia + estacionalidad aditivas (m = 4)"
    else:
        modelo = ExponentialSmoothing(y, trend="add", seasonal=None, initialization_method="estimated")
        especificacion = "simplificado: nivel + tendencia"
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        ajuste = modelo.fit(optimized=True)
    registro_ajustes.append({"etapa": etiqueta.split(" · ")[0], "ajuste": etiqueta, "n_train": len(y),
                             "especificacion": especificacion, "advertencias": len(avisos),
                             "tipos": "; ".join(sorted({w.category.__name__ for w in avisos})),
                             "mensajes": " | ".join(sorted({str(w.message)[:110] for w in avisos}))})
    return ajuste

def pronosticar(modelo, y, h, etiqueta=""):
    if modelo == "Naive":
        f = np.repeat(y.iloc[-1], h)
    elif modelo == "SeasonalNaive":
        f = np.resize(y.iloc[-SEASON:].values, h)
    elif modelo == "ExpSmoothing":
        f = np.asarray(ajustar_es(y, etiqueta).forecast(h), dtype=float)
    else:
        raise ValueError(modelo)
    return np.maximum(np.round(f), 0)   # cantidad operativa

def evaluar(real, cantidad, over=OVER_COST, short=SHORT_COST):
    real, cantidad = np.asarray(real, float), np.asarray(cantidad, float)
    exceso, faltante = np.maximum(cantidad - real, 0), np.maximum(real - cantidad, 0)
    return {"MAE": mean_absolute_error(real, cantidad),
            "RMSE": mean_squared_error(real, cantidad) ** 0.5,
            "WAPE": np.abs(real - cantidad).sum() / real.sum() if real.sum() > 0 else np.nan,  # no definido si la demanda total es 0
            "unidades_excedentes": exceso.sum(), "unidades_faltantes": faltante.sum(),
            "costo": (exceso * over + faltante * short).sum()}

def resumir(filas, por, orden):
    """Resume los folds por modelo o por política. 'orden' va de la opción más simple a la más compleja."""
    t = (pd.DataFrame(filas).groupby(por)
           .agg(costo_prom=("costo", "mean"), costo_sd=("costo", "std"),
                MAE_prom=("MAE", "mean"), MAE_sd=("MAE", "std"), RMSE_prom=("RMSE", "mean"),
                WAPE_prom=("WAPE", "mean"), excedentes=("unidades_excedentes", "sum"),
                faltantes=("unidades_faltantes", "sum"))
           .reindex(orden))
    t["complejidad"] = range(len(orden))
    return t

def seleccionar(tabla, tol=TOLERANCIA):
    """Regla única de selección (ver el texto de esta sección)."""
    minimo = tabla["costo_prom"].min()
    limite = minimo * (1 + tol)          # si minimo == 0, limite == 0: solo empatan costos 0
    candidatos = tabla[tabla["costo_prom"] <= limite + 1e-9]
    return candidatos.sort_values(["MAE_prom", "costo_sd", "complejidad"], kind="stable").index[0]
print("Funciones listas.")

Funciones listas.


# 9. Backtesting rolling-origin (solo con el entrenamiento)

Cada fold entrena con todo lo anterior a su origen y pronostica las **8 semanas** siguientes, igual que el test final. Se exige al menos 52 semanas de entrenamiento; los bloques de validación no se superponen y el último termina justo antes del test. La celda **comprueba** que ningún fold invada el test.

In [11]:
BT_H, MIN_TRAIN = 8, 52
origenes = list(range(len(train) - BT_H, MIN_TRAIN - 1, -BT_H))[::-1]

def backtest(y, modelos, etiqueta):
    """Pronostica cada bloque de validación desde su origen. Devuelve métricas por fold y cantidades."""
    filas, cantidades = [], {}
    for k, origen in enumerate(origenes, start=1):
        tr, va = y.iloc[:origen], y.iloc[origen:origen + BT_H]
        assert tr.index[-1] < va.index[0] and va.index[-1] < semanas_test[0], "un fold invade el test"
        for m in modelos:
            cantidades[(k, m)] = pronosticar(m, tr, BT_H, f"{etiqueta} · fold {k}")
            filas.append({**evaluar(va, cantidades[(k, m)]), "fold": k, "modelo": m})
    return filas, cantidades

filas_bt, q_bt = backtest(train, MODELOS, "backtesting")
bt = pd.DataFrame(filas_bt)
folds = pd.DataFrame([{"fold": k, "entrenamiento": f"{train.index[0].date()} → {train.index[o - 1].date()}", "n_train": o,
                       "validacion": f"{train.index[o].date()} → {train.index[o + BT_H - 1].date()}"}
                      for k, o in enumerate(origenes, start=1)])
display(folds)
display(bt.round(2))
print("Comprobado: los", len(origenes), "folds terminan antes del test (", semanas_test[0].date(), ").")

,fold,entrenamiento,n_train,validacion
0,1,2009-12-07 → 2010-12-27,56,2011-01-03 → 2011-02-21
1,2,2009-12-07 → 2011-02-21,64,2011-02-28 → 2011-04-18
2,3,2009-12-07 → 2011-04-18,72,2011-04-25 → 2011-06-13
3,4,2009-12-07 → 2011-06-13,80,2011-06-20 → 2011-08-08
4,5,2009-12-07 → 2011-08-08,88,2011-08-15 → 2011-10-03


,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo,fold,modelo
0,608.12,1044.62,1.00,0.0,4865.0,7297.5,1,Naive
1,528.88,758.45,0.87,1195.0,3036.0,5032.0,1,SeasonalNaive
2,1599.12,1838.12,2.63,11475.0,1318.0,6567.0,1,ExpSmoothing
3,1111.12,1460.84,0.83,113.0,8776.0,13209.2,2,Naive
4,789.88,1000.51,0.59,1317.0,5002.0,8029.8,2,SeasonalNaive
5,949.12,1147.68,0.71,4251.0,3342.0,6713.4,2,ExpSmoothing
6,723.88,824.92,0.77,4550.0,1241.0,3681.5,3,Naive
7,1248.88,1463.50,1.33,8557.0,1434.0,5573.8,3,SeasonalNaive
8,850.75,960.03,0.91,5384.0,1422.0,4286.6,3,ExpSmoothing
9,391.88,471.10,0.75,47.0,3088.0,4650.8,4,Naive


Comprobado: los 5 folds terminan antes del test ( 2011-10-10 ).


In [12]:
resumen_val = resumir(filas_bt, "modelo", MODELOS)
SELECCIONADO = seleccionar(resumen_val)
display(resumen_val.round(2))
print("MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test):", SELECCIONADO)
fig = px.box(bt, x="modelo", y="costo", points="all", title="Costo simulado por fold de validación (£)",
             labels={"modelo": "Modelo", "costo": "Costo por fold (£)"})
fig.show()

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
modelo,,,,,,,,,
Naive,6239.22,4296.41,620.2,327.68,819.13,0.75,5469.0,19339.0,0
SeasonalNaive,4950.34,2065.83,666.5,370.02,830.61,0.83,13853.0,12807.0,1
ExpSmoothing,4324.10,2335.11,927.3,420.72,1063.63,1.29,30925.0,6167.0,2


MODELO SELECCIONADO EN VALIDACIÓN (fijado antes de mirar el test): ExpSmoothing


### Pregunta 4 · Modelo que elegiría usando solo la validación

La regla de costo medio con tolerancia del 1 % elige **Exponential Smoothing**: obtiene £4 324,10. El límite
admisible queda en £4 367,34 y los otros dos modelos están por encima (Naïve estacional £4 950,34; Naïve
£6 239,22). Por eso esta elección no necesita aplicar un desempate.

Un detalle importante es que **el ranking económico contradice al del MAE y RMSE**. Naïve registra un MAE
medio de 620,2, mientras que Exponential Smoothing alcanza 927,3; aun así, el modelo de suavizamiento tiene
menor costo porque tiende a planificar más unidades y evita faltantes, que cuestan £1,50 cada uno. Como
referencia, en validación Exponential Smoothing deja 6 167 faltantes contra 19 339 del Naïve.

El **WAPE no añade una forma diferente de ordenar** estos modelos cuando, dentro de cada fold, se compara
contra el mismo volumen real: utiliza los mismos errores absolutos que MAE, pero divididos entre la misma suma
de unidades observadas. Mi conclusión es que evaluaría el método mediante **costo + error + sesgo**, no solo
con una métrica estadística.


### Pregunta 5 · ¿La ventaja económica es estable?

Si comparo la dispersión entre los cinco folds, **Naïve** es el menos estable económicamente: su costo tiene
una desviación estándar de **£4 296,41**. El más estable resulta **Naïve estacional** con **£2 065,83**, y
**Exponential Smoothing** queda al medio con **£2 335,11**. En términos de MAE, en cambio, el suavizamiento
es el más variable (desviación **420,72**), frente a **327,68** del Naïve.

El modelo elegido superó económicamente al Naïve en cuatro de cinco folds, pero no en todos. Para mí esa es
la señal de advertencia: cuando una demanda cambia de manera brusca, ganar en promedio puede ocultar un
periodo de pérdida importante. Si esto representara repuestos críticos, pediría una simulación por periodos
de alta demanda antes de autorizar un cambio en abastecimiento.


# 10. Política de colchón calibrada sin información futura (antes del test)

El colchón suma unidades al pronóstico para protegerse de faltantes: **colchón = ⌈multiplicador × MAE previo⌉**. Se comparan los multiplicadores 0; 0,5; 1 y 1,5.

**Regla de calibración (sin errores futuros).** Para cada origen, el *MAE previo* se calcula con el modelo seleccionado en los **2 bloques de 8 semanas inmediatamente anteriores** a ese origen: cada bloque se pronostica entrenando solo con lo anterior a él. Así, el colchón del fold 1 usa solo errores anteriores al fold 1.
- Si no hay historia suficiente (algún bloque quedaría con menos de 16 semanas de entrenamiento), el colchón de ese origen es **0** y se informa. Con 52 semanas mínimas no ocurre en este laboratorio.
- La política (modelo, multiplicador y regla) se elige con la **regla única de selección** y queda **fijada antes del test**.
- Para el test, el MAE previo se calcula con la misma regla desde el final del entrenamiento (solo información previa al test).
- Después de evaluar, la **magnitud** del colchón se actualiza con la historia completa para proyectar, **sin cambiar la regla**.

In [13]:
MULTIPLICADORES = [0, 0.5, 1.0, 1.5]
BLOQUES_CAL = 2

def mae_previo(modelo, y, etiqueta, bloques=BLOQUES_CAL):
    """MAE del modelo en los 'bloques' bloques de 8 semanas que terminan al final de y.
    Solo usa y (información hasta el origen). Devuelve None si la historia no alcanza."""
    errores = []
    for b in range(bloques, 0, -1):
        corte = len(y) - b * BT_H
        if corte < 2 * SEASON + 8:
            return None
        real = y.iloc[corte:corte + BT_H].values
        errores.append(np.abs(real - pronosticar(modelo, y.iloc[:corte], BT_H, f"{etiqueta} · bloque {b}")))
    return float(np.mean(np.concatenate(errores)))

def colchon(mult, mae):
    return 0 if (mae is None or mult == 0) else int(np.ceil(mult * mae))

def comparar_politicas(bloques=BLOQUES_CAL, etiqueta="calibración"):
    filas = []
    for k, origen in enumerate(origenes, start=1):
        tr, va = train.iloc[:origen], train.iloc[origen:origen + BT_H]
        mae_k = mae_previo(SELECCIONADO, tr, f"{etiqueta} · fold {k}", bloques)
        for mult in MULTIPLICADORES:
            c = colchon(mult, mae_k)
            filas.append({**evaluar(va, q_bt[(k, SELECCIONADO)] + c), "fold": k, "multiplicador": mult,
                          "mae_previo": mae_k, "colchon": c,
                          "ultima_semana_calibracion": tr.index[-1].date(), "inicio_validacion": va.index[0].date()})
    return pd.DataFrame(filas)

pol_folds = comparar_politicas()
assert (pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all(), "colchón con información futura"
resumen_pol = resumir(pol_folds, "multiplicador", MULTIPLICADORES)
MULT = seleccionar(resumen_pol)

display(pol_folds.loc[pol_folds["multiplicador"] == 1.0,
                      ["fold", "ultima_semana_calibracion", "inicio_validacion", "mae_previo"]].round(1))
display(resumen_pol.round(2))
MAE_TEST = mae_previo(SELECCIONADO, train, "calibración test")       # solo entrenamiento
COLCHON_TEST = colchon(MULT, MAE_TEST)
POLITICA = f"{SELECCIONADO} + {MULT:g} × MAE previo"
print("POLÍTICA FIJADA ANTES DEL TEST:", POLITICA)
print(f"Colchón para el test: {COLCHON_TEST} unidades por semana (MAE previo = {MAE_TEST:.1f}, calculado solo con el entrenamiento)")

,fold,ultima_semana_calibracion,inicio_validacion,mae_previo
2,1,2010-12-27,2011-01-03,1400.7
6,2,2011-02-21,2011-02-28,1533.2
10,3,2011-04-18,2011-04-25,1274.1
14,4,2011-06-13,2011-06-20,899.9
18,5,2011-08-08,2011-08-15,821.3


,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad
multiplicador,,,,,,,,,
0.0,4324.10,2335.11,927.30,420.72,1063.63,1.29,30925.0,6167.0,0
0.5,4806.72,1784.08,1334.35,475.19,1477.53,1.88,50934.0,2440.0,1
1.0,5833.74,1661.61,1812.80,521.90,1988.40,2.53,72363.0,149.0,2
1.5,7674.08,2000.88,2398.15,625.28,2537.15,3.31,95926.0,0.0,3


POLÍTICA FIJADA ANTES DEL TEST: ExpSmoothing + 0 × MAE previo
Colchón para el test: 0 unidades por semana (MAE previo = 618.8, calculado solo con el entrenamiento)


### Pregunta 6 · Colchón de seguridad seleccionado

Seleccioné con datos anteriores al test el **multiplicador 0**, por lo que no agrego unidades al pronóstico.
En la validación, esa política cuesta **£4 324,10** por fold en promedio. Con colchones 0,5×, 1× y 1,5×
el costo aumenta a **£4 806,72**, **£5 833,74** y **£7 674,08**, respectivamente. La calibración para
el test usa MAE histórico de **618,8 unidades**, pero con multiplicador cero el colchón sigue siendo cero.

Un margen de seguridad no es gratis: puede reducir el faltante, pero en este caso aumenta demasiado las
unidades que sobran. Además, el MAE utilizado en cada fold se calcula con errores **anteriores a su fecha de
inicio**. Si lo calculara usando la misma ventana a pronosticar, estaría aprovechando datos futuros y
construyendo una política que no podría repetir en una operación real.


# 11. Evaluación final en test (una sola vez)

Se reentrena cada modelo con las 96 semanas de entrenamiento y se pronostican las 8 semanas de test. Se evalúan los tres modelos sin colchón y la **política fijada**. Si otra opción resulta mejor aquí, se informa, pero **no** cambia la selección.

In [14]:
cantidades_test = {m: pronosticar(m, train, TEST_H, "test") for m in MODELOS}
q_politica = cantidades_test[SELECCIONADO] + COLCHON_TEST
filas = [{**evaluar(test, q), "opcion": f"{m} (sin colchón)"} for m, q in cantidades_test.items()]
if MULT > 0:
    filas.append({**evaluar(test, q_politica), "opcion": f"Política fijada: {POLITICA}"})
res_test = pd.DataFrame(filas).set_index("opcion")
display(res_test.round(2))

fila_pol = f"Política fijada: {POLITICA}" if MULT > 0 else f"{SELECCIONADO} (sin colchón)"
costo_pol, costo_naive = res_test.loc[fila_pol, "costo"], res_test.loc["Naive (sin colchón)", "costo"]
diferencia = costo_naive - costo_pol        # > 0 ahorro; < 0 sobrecosto
print("Política evaluada:", POLITICA)
if SELECCIONADO == "Naive" and MULT == 0:
    print("La política seleccionada es el propio Naïve: no hay diferencia que calcular frente a él.")
elif costo_naive == 0:
    print(f"Diferencia frente a Naïve: £{diferencia:.2f}. No se calcula porcentaje porque el costo de Naïve es cero.")
elif diferencia > 0:
    print(f"Ahorro frente a Naïve: £{diferencia:.2f} ({diferencia / costo_naive * 100:.1f} %).")
else:
    print(f"Sobrecosto frente a Naïve: £{-diferencia:.2f} ({-diferencia / costo_naive * 100:.1f} %).")
mejor_retro = res_test["costo"].idxmin()
if mejor_retro != fila_pol:
    print(f"Retrospectivamente, «{mejor_retro}» habría costado menos en test; la selección no cambia.")

fig = go.Figure()
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode="lines+markers", name="Ventas reales", line=dict(width=3)))
for m, q in cantidades_test.items():
    fig.add_trace(go.Scatter(x=test.index, y=q, mode="lines+markers", name=m))
fig.update_layout(title="Test final: ventas reales y cantidades operativas", xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
opcion,,,,,,
Naive (sin colchón),678.75,1545.12,0.48,961.0,4469.0,7087.9
SeasonalNaive (sin colchón),773.75,1631.81,0.54,641.0,5549.0,8579.9
ExpSmoothing (sin colchón),776.62,1700.86,0.54,329.0,5884.0,8957.6


Política evaluada: ExpSmoothing + 0 × MAE previo
Sobrecosto frente a Naïve: £1869.70 (26.4 %).
Retrospectivamente, «Naive (sin colchón)» habría costado menos en test; la selección no cambia.


### Pregunta 7 · Comprobación única con el test

**La política elegida no consiguió superar el método sencillo.** En las ocho semanas apartadas, el costo de
Exponential Smoothing sin colchón fue **£8 957,60**. El Naïve estacional quedó en **£8 579,90** y el Naïve
simple en **£7 087,90**. Eso significa pagar **£1 869,70 adicionales**, aproximadamente 26,4 %, frente
a la referencia.

En el test el Naïve también logró el menor RMSE (1 545,12). Lo que me interesa no es buscar otro parámetro
mirando el test, sino reconocer que la ventaja observada durante validación no se sostuvo en fechas nuevas.
Mantendría el Naïve como referencia y continuaría observando el suavizamiento sin usarlo para ordenar compras.


### Diagnóstico del pico del test (retrospectivo)

Se identifica la semana de mayor venta del test y el pedido más grande de esa semana. Un pedido extraordinario **aislado** es un **pico puntual**; un **quiebre estructural** exige un cambio persistente de nivel o patrón, que 8 semanas no permiten confirmar. Los modelos usados son **univariantes**: solo ven las ventas pasadas del SKU y no disponían de información específica sobre ese pedido.

La evaluación principal **conserva** esa semana. El cálculo sin ella es un **análisis retrospectivo de sensibilidad**, no una evaluación alternativa.

In [15]:
semana_pico = test.idxmax()
lineas_pico = uk[(uk["StockCode"] == SKU) & (uk["week"] == semana_pico)]
pedidos = lineas_pico.groupby(["InvoiceNo", "CustomerID"], dropna=False)["Quantity"].sum().sort_values(ascending=False)
print(f"Semana de mayor venta en test: {semana_pico.date()} · {test.max():.0f} unidades")
factura, cliente = pedidos.index[0]
cliente = "sin dato" if pd.isna(cliente) else str(cliente).removesuffix(".0")
print(f"Pedido más grande: factura {factura} · cliente {cliente} · {pedidos.iloc[0]:.0f} unidades "
      f"({pedidos.iloc[0] / test.max() * 100:.1f} % de la semana)")
print("Mediana semanal del entrenamiento:", train.median(), "unidades")

sin_pico = test.index != semana_pico
sens = pd.DataFrame({m: evaluar(test[sin_pico], cantidades_test[m][sin_pico]) for m in MODELOS}).T
if MULT > 0:
    sens.loc["Política fijada"] = evaluar(test[sin_pico], q_politica[sin_pico])
print("\nANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.")
display(sens[["MAE", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))

Semana de mayor venta en test: 2011-10-24 · 5328 unidades
Pedido más grande: factura 573008 · cliente 12901 · 4800 unidades (90.1 % de la semana)
Mediana semanal del entrenamiento: 584.5 unidades

ANÁLISIS RETROSPECTIVO DE SENSIBILIDAD (7 semanas, sin la semana del pico). No reemplaza la evaluación principal.


,MAE,unidades_excedentes,unidades_faltantes,costo
Naive,155.86,961.0,130.0,579.4
SeasonalNaive,232.86,641.0,989.0,1739.9
ExpSmoothing,205.86,329.0,1112.0,1799.6


### Pregunta 8 · Diagnóstico del error y del pedido extraordinario

El riesgo predominante fue **quedarse corto**. La política elegida produjo 5 884 unidades faltantes
(£8 826,00) y 329 sobrantes (£131,60). El 98,5 % del costo simulado proviene de faltantes, pese a que el
modelo tendía a sobreestimar durante la validación. Esa inversión demuestra por qué no bastaba el backtesting.

La semana del **24 de octubre de 2011** registra un pico de **5 328 unidades**. Al revisar las facturas,
**4 800 unidades (90,1 %)** corresponden a un único pedido; sin esa operación, la semana habría mostrado
aproximadamente 528 unidades. Por eso lo describo como **pico puntual**, no como quiebre estructural
confirmado: un cambio permanente requeriría muchas más observaciones posteriores.

Eliminar esa semana solo como análisis retrospectivo tampoco favorece al modelo seleccionado. En las otras
siete semanas Naïve conserva el costo más bajo (**£579,40** frente a **£1 799,60** de Exponential
Smoothing). No puedo usar esa exclusión para cambiar la selección; solo sirve para entender la exposición
a pedidos grandes. En un negocio de equipos y repuestos esta señal me llevaría a consultar ventas corporativas
y pedidos programados, no a corregir el resultado hasta hacerlo favorable.


# 12. Reentrenamiento final y proyección académica

Después de cerrar la evaluación se reentrena el modelo seleccionado con **toda** la historia (104 semanas) y se actualiza la **magnitud** del colchón con la misma regla (los 2 bloques más recientes, que ahora incluyen el test). Se muestra también la proyección de Naïve como referencia.

La tabla es una **proyección académica**, no una orden de compra:
- La **cantidad objetivo semanal** es un objetivo para cada semana. El laboratorio **no simula el arrastre de inventario** entre semanas: si una semana sobra mercadería, esta reduce la necesidad de la siguiente. Por eso la suma de los objetivos semanales no equivale a la cantidad a comprar.
- Una compra simplificada para una semana sería `max(objetivo − inventario utilizable − pedidos que llegarán a tiempo, 0)`: nunca negativa, y solo cuenta el inventario apto para la venta y los pedidos que llegarán antes de esa semana. El laboratorio no dispone de esos datos.

In [16]:
FINAL_H = 8
pron_final = pronosticar(SELECCIONADO, serie, FINAL_H, "final")
pron_naive = pronosticar("Naive", serie, FINAL_H, "final")
MAE_FINAL = mae_previo(SELECCIONADO, serie, "calibración final")   # misma regla, historia completa
COLCHON_FINAL = colchon(MULT, MAE_FINAL)
semanas_futuras = pd.date_range(serie.index[-1] + pd.Timedelta(weeks=1), periods=FINAL_H, freq="W-MON")
plan = pd.DataFrame({"Semana": semanas_futuras.date,
                     f"Pronóstico {SELECCIONADO}": pron_final.astype(int),
                     "Colchón (política fijada)": COLCHON_FINAL,
                     "Cantidad objetivo semanal": (pron_final + COLCHON_FINAL).astype(int),
                     "Referencia Naïve": pron_naive.astype(int)})
display(plan)
print(f"Política aplicada: {POLITICA} · MAE previo actualizado = {MAE_FINAL:.1f} · colchón = {COLCHON_FINAL} unidades por semana")
print("Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.")

,Semana,Pronóstico ExpSmoothing,Colchón (política fijada),Cantidad objetivo semanal,Referencia Naïve
0,2011-12-05,1075,0,1075,689
1,2011-12-12,899,0,899,689
2,2011-12-19,1617,0,1617,689
3,2011-12-26,998,0,998,689
4,2012-01-02,1103,0,1103,689
5,2012-01-09,927,0,927,689
6,2012-01-16,1645,0,1645,689
7,2012-01-23,1027,0,1027,689


Política aplicada: ExpSmoothing + 0 × MAE previo · MAE previo actualizado = 611.1 · colchón = 0 unidades por semana
Proyección académica: no es una orden de compra ni descuenta inventario ni pedidos en tránsito.


# 13. Decisión de adopción y resumen automático

La decisión se toma con una regla fijada de antemano:
- **Adoptar** (como piloto controlado): la política seleccionada costó menos que Naïve en el test **y** en la mayoría de los folds de validación.
- **Mantener en evaluación:** superó a Naïve en el test, pero no de forma consistente en validación.
- **No adoptar como sustituto de Naïve:** en el test costó igual o más que Naïve. Naïve sigue como referencia; el modelo puede seguir evaluándose sin decidir compras con él.

Un **piloto** sirve para comprobar, durante varias semanas nuevas y con la misma regla, si la política reduce el costo frente a Naïve; no justifica por sí mismo una cantidad de compra.

In [17]:
naive_bt = bt[bt["modelo"] == "Naive"].set_index("fold")["costo"]
pol_bt = pol_folds[pol_folds["multiplicador"] == MULT].set_index("fold")["costo"]
folds_ganados = int((pol_bt < naive_bt).sum())
if costo_pol < costo_naive and folds_ganados > len(origenes) / 2:
    DECISION = "Adoptar como piloto controlado frente a Naïve"
elif costo_pol < costo_naive:
    DECISION = "Mantener en evaluación"
else:
    DECISION = "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

resumen = {
    "Producto analizado": f"{SKU} · {desc}",
    "Seleccionado antes del test": f"{SELECCIONADO}; política {POLITICA} (regla única de selección)",
    "Costo promedio de validación de la política": f"£{resumen_pol.loc[MULT, 'costo_prom']:.2f} ± £{resumen_pol.loc[MULT, 'costo_sd']:.2f} ({len(origenes)} folds)",
    "Folds de validación en que superó a Naïve": f"{folds_ganados} de {len(origenes)}",
    "Resultado en test (8 semanas)": f"£{costo_pol:.2f} frente a £{costo_naive:.2f} de Naïve",
    "Frente a Naïve": ("ahorro" if diferencia > 0 else "sobrecosto") + f" de £{abs(diferencia):.2f}" + (f" ({abs(diferencia) / costo_naive * 100:.1f} %)" if costo_naive > 0 else ""),
    "Unidades excedentes / faltantes en test": f"{res_test.loc[fila_pol, 'unidades_excedentes']:.0f} / {res_test.loc[fila_pol, 'unidades_faltantes']:.0f}",
    "Decisión": DECISION,
    "Proyección académica (suma de objetivos semanales)": f"{int(plan['Cantidad objetivo semanal'].sum())} unidades; no es una orden de compra",
}
for k, v in resumen.items():
    print(f"{k}: {v}")

Producto analizado: 84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS
Seleccionado antes del test: ExpSmoothing; política ExpSmoothing + 0 × MAE previo (regla única de selección)
Costo promedio de validación de la política: £4324.10 ± £2335.11 (5 folds)
Folds de validación en que superó a Naïve: 4 de 5
Resultado en test (8 semanas): £8957.60 frente a £7087.90 de Naïve
Frente a Naïve: sobrecosto de £1869.70 (26.4 %)
Unidades excedentes / faltantes en test: 329 / 5884
Decisión: No adoptar como sustituto de Naïve; Naïve se mantiene como referencia
Proyección académica (suma de objetivos semanales): 9291 unidades; no es una orden de compra


### Pregunta 9 — Mi informe breve para un responsable de operaciones
Cierre ejecutivo (máximo 12 líneas). Distinga: modelo y política seleccionados antes del test; resultado de su evaluación frente a Naïve; decisión (adoptar, mantener en evaluación o no adoptar) y su fundamento; proyección académica y por qué no es una orden de compra; una condición que cambiaría la decisión; y las limitaciones (supuestos de costos, ceros, disponibilidad y un solo test de 8 semanas).

**Respuesta — análisis realizado con los supuestos del laboratorio:**
1. **Modelo escogido sin mirar el test:** Exponential Smoothing (nivel + tendencia + estacionalidad aditiva, m = 4) con la política sin colchón (0 × MAE previo); costo promedio de validación de £4 324,10 ± £2 335,11 y menor costo que Naïve en 4 de 5 folds.
2. **Resultado en test (8 semanas):** costó £8 957,60 frente a £7 087,90 de Naïve, un sobrecosto de £1 869,70 (26,4 %), con 5 884 unidades faltantes y 329 excedentes.
3. **Mi decisión: no sustituir el Naïve todavía.** La regla exige costar menos que Naïve en el test y no se cumplió; Naïve queda como referencia y el modelo puede seguir evaluándose sin decidir compras con él.
4. **Volumen planificado, no compra automática:** 9 291 unidades en 8 semanas (entre 899 y 1 645 por semana). No es una orden de compra: no descuenta inventario utilizable ni pedidos en tránsito, ni el arrastre de excedentes entre semanas.
5. **Qué cambiaría la decisión:** que, en varias semanas nuevas evaluadas con la misma regla, el modelo cueste menos que Naïve de forma consistente, o que el área comercial anticipe los pedidos grandes de clientes mayoristas.
6. **Limitaciones:** los costos (£1,50 y £0,40) son supuestos académicos; los ceros no distinguen demanda nula de falta de inventario; no se conoce la disponibilidad real; y el test es uno solo, de 8 semanas, dominado por un pedido de 4 800 unidades.


# 14. Retos individuales de aplicación

La parte guiada ya desarrolló los procedimientos. En esta sección los aplicará a tres casos empresariales. Cada reto tiene cuatro apartados: **A** plantear o preparar el análisis, **B** implementar y obtener resultados, **C** comparar e interpretar la evidencia y **D** formular una decisión justificada con sus limitaciones. Reutilice las funciones de la parte guiada (`evaluar`, `resumir`, `seleccionar`, `backtest`, `pronosticar`, `mae_previo`, `colchon` y `comparar_politicas`) y los objetos ya calculados (`train`, `test`, `origenes`, `q_bt`, `cantidades_test`, `elegibles`). No modifique datos, semanas, pronósticos ni el test. Las respuestas deben ser breves, pero suficientes para justificar la decisión con cifras.

> **Entrega.** «Resuelva los tres retos y sus apartados A, B, C y D en su notebook personal. Guarde la versión ejecutada, con resultados e interpretaciones, en la carpeta semana10/ de su repositorio personal de GitHub y en su propio Google Colab. En la actividad de la semana, en el espacio habilitado, presente ambos enlaces: GitHub y Colab. Verifique que permitan la revisión del trabajo.»

## Reto 1 — Una campaña cambia el costo de los faltantes

**Caso.** La empresa prepara una campaña comercial. Durante la campaña, quedarse sin producto significa perder ventas y clientes que llegaron por la promoción, mientras que guardar unidades sobrantes cuesta menos. El responsable de abastecimiento necesita saber si la recomendación del modelo cambia cuando el faltante se vuelve más costoso.

**Problema.** ¿Se elegiría el mismo método de pronóstico bajo distintos costos del error y, con la evidencia del test, cuál debería usar abastecimiento como referencia?

**Datos, supuestos y recursos**
- Producto y semanas del laboratorio; pronósticos de validación ya calculados (`q_bt`) y cantidades de test (`cantidades_test`).
- Escenarios de costo (supuestos académicos, no provienen de UCI): A faltante £1,00 y exceso £0,50; B faltante £1,50 y exceso £0,40 (base); C faltante £3,00 y exceso £0,25.
- Regla común de selección (`resumir` y `seleccionar`). No se modifican ventas ni pronósticos para favorecer un resultado.

**Apartados**
- **A.** Plantear los escenarios y la decisión
- **B.** Calcular los costos de validación y seleccionar
- **C.** Evaluar en test y comparar frente a Naïve
- **D.** Recomendar a abastecimiento

**Evidencias que deben quedar en el notebook**
- Tabla de escenarios con la razón faltante / exceso.
- Tabla de costos promedio de validación por escenario y modelo, con el seleccionado.
- Tabla de test con excedentes, faltantes, costo y diferencia frente a Naïve, y su gráfico.
- Interpretaciones A–D.

### Reto 1 · Apartado A — Plantear los escenarios y la decisión
Construya una tabla con los tres escenarios y la razón faltante / exceso. Explique qué representa cada escenario para el negocio y formule la decisión empresarial que se tomará con el análisis.

In [18]:
ESCENARIOS = {"A": (1.00, 0.50), "B": (1.50, 0.40), "C": (3.00, 0.25)}   # (faltante, exceso) en £ por unidad
# Tabla de escenarios: costo por unidad faltante, por unidad excedente y cuántas veces pesa más el faltante
r1_escenarios = pd.DataFrame([{"escenario": esc, "faltante": short, "exceso": over, "faltante / exceso": short / over}
                              for esc, (short, over) in ESCENARIOS.items()]).set_index("escenario")
display(r1_escenarios)

,faltante,exceso,faltante / exceso
escenario,,,
A,1.0,0.50,2.00
B,1.5,0.40,3.75
C,3.0,0.25,12.00


**Interpretación 1A · Mis supuestos de costo:**

Comparo **tres situaciones**. En la A (£1,00 por falta y £0,50 por exceso) la escasez cuesta el doble que
la acumulación; en la B (£1,50 y £0,40) cuesta 3,75 veces; en la C (£3,00 y £0,25), doce veces. Por eso C
representa un contexto donde perder una venta es especialmente caro, como una campaña o una venta con
compromisos de entrega. Mi pregunta no es cuál modelo pronostica mejor en abstracto, sino **cuál minimiza el
costo bajo el escenario previsto**.


### Reto 1 · Apartado B — Calcular los costos de validación y seleccionar
Para cada escenario, recalcule el costo de cada fold con los pronósticos ya disponibles (`q_bt`), resuma por modelo y aplique la regla común de selección. Presente el costo promedio de validación de los tres modelos y el seleccionado en cada escenario.

In [19]:
r1_val, r1_eleccion = {}, {}
for esc, (short, over) in ESCENARIOS.items():
    # Mismos pronósticos de validación (q_bt); solo cambian los costos unitarios del escenario
    filas = [{**evaluar(train.iloc[o:o + BT_H], q_bt[(k, m)], over=over, short=short), "fold": k, "modelo": m}
             for k, o in enumerate(origenes, start=1) for m in MODELOS]
    r1_val[esc] = resumir(filas, "modelo", MODELOS)
    r1_eleccion[esc] = seleccionar(r1_val[esc])
# Costo promedio de validación por escenario y modelo, con el modelo que elige la regla común
tabla_r1b = pd.DataFrame({esc: r1_val[esc]["costo_prom"] for esc in ESCENARIOS}).T
tabla_r1b.index.name = "escenario"
tabla_r1b["seleccionado"] = pd.Series(r1_eleccion)
tabla_r1b["candidatos (≤ mínimo × 1,01)"] = [", ".join(r1_val[e].index[r1_val[e]["costo_prom"] <= r1_val[e]["costo_prom"].min() * (1 + TOLERANCIA) + 1e-9])
                                            for e in ESCENARIOS]
display(tabla_r1b.round(2))

modelo,Naive,SeasonalNaive,ExpSmoothing,seleccionado,"candidatos (≤ mínimo × 1,01)"
escenario,,,,,
A,4414.70,3946.70,4325.90,SeasonalNaive,SeasonalNaive
B,6239.22,4950.34,4324.10,ExpSmoothing,ExpSmoothing
C,11876.85,8376.85,5246.45,ExpSmoothing,ExpSmoothing


**Interpretación 1B · Qué cambia en la validación:**

Cuando el faltante tiene peso moderado (A), elegiría **Naïve estacional** con costo medio **£3 946,70**.
Con el escenario B el ganador cambia a **Exponential Smoothing (£4 324,10)** y en C se mantiene ese modelo
(**£5 246,45**). Ningún segundo candidato queda dentro del margen de tolerancia de 1 %.

Mi lectura es que Exponential Smoothing genera más excedentes, pero durante validación logra muchos menos
faltantes. Por eso, cuando el precio de la escasez aumenta, la ecuación económica premia ese comportamiento.
**No significa** que ese patrón necesariamente vaya a continuar en el periodo de prueba.


### Reto 1 · Apartado C — Evaluar en test y comparar frente a Naïve
Evalúe en test el modelo seleccionado en cada escenario (la selección ya quedó definida con validación) y Naïve con los mismos costos. Construya una tabla con excedentes, faltantes, costo y diferencia frente a Naïve, y un gráfico de barras. Interprete qué escenario y qué tipo de error explican las diferencias.

In [20]:
# Se evalúa en test el modelo ya elegido en validación para cada escenario, junto con Naïve, con los mismos costos
filas = []
for esc, (short, over) in ESCENARIOS.items():
    sel = r1_eleccion[esc]
    ev_sel = evaluar(test, cantidades_test[sel], over=over, short=short)
    ev_naive = evaluar(test, cantidades_test["Naive"], over=over, short=short)
    filas.append({"escenario": esc, "seleccionado": sel,
                  "excedentes": ev_sel["unidades_excedentes"], "faltantes": ev_sel["unidades_faltantes"],
                  "costo_test": ev_sel["costo"], "costo_test_Naive": ev_naive["costo"],
                  "diferencia_vs_Naive": ev_naive["costo"] - ev_sel["costo"]})   # > 0 ahorro; < 0 sobrecosto
tabla_r1c = pd.DataFrame(filas).set_index("escenario")
display(tabla_r1c.round(2))

etiquetas = [f"{esc} · {tabla_r1c.loc[esc, 'seleccionado']}" for esc in tabla_r1c.index]
fig = go.Figure([go.Bar(x=etiquetas, y=tabla_r1c["costo_test"], name="Modelo seleccionado"),
                 go.Bar(x=etiquetas, y=tabla_r1c["costo_test_Naive"], name="Naïve")])
fig.update_layout(barmode="group", title="Reto 1 · Costo en test por escenario (£)",
                  xaxis_title="Escenario · modelo seleccionado", yaxis_title="Costo en test (£)")
fig.show()

,seleccionado,excedentes,faltantes,costo_test,costo_test_Naive,diferencia_vs_Naive
escenario,,,,,,
A,SeasonalNaive,641.0,5549.0,5869.50,4949.50,-920.0
B,ExpSmoothing,329.0,5884.0,8957.60,7087.90,-1869.7
C,ExpSmoothing,329.0,5884.0,17734.25,13647.25,-4087.0


**Interpretación 1C · Verificación con las semanas reservadas:**

La comparación final no valida ninguno de los modelos seleccionados frente a Naïve. En A el Naïve estacional
cuesta **£5 869,50** contra **£4 949,50** de Naïve; en B, Exponential Smoothing cuesta **£8 957,60**
contra **£7 087,90**; y en C, **£17 734,25** contra **£13 647,25**. El sobrecosto aumenta en C porque
se penalizan más las unidades no cubiertas. Concluiría que los cambios de costos **sí modifican qué modelo
resulta atractivo en validación**, pero no convierten ese resultado en una ventaja real en el test.


### Reto 1 · Apartado D — Recomendar a abastecimiento
Redacte una recomendación para abastecimiento: ¿cambia la selección entre escenarios?, ¿qué método usaría como referencia y por qué?, ¿qué límites tiene la conclusión? Distinga el ahorro simulado de un beneficio real demostrado.

**Decisión 1D · Mi propuesta para abastecimiento:**

Mantendría **Naïve** como estimación de referencia mientras continúe la evaluación. Aunque la campaña haga
más costoso quedarse sin productos, Exponential Smoothing generó un sobrecosto simulado de **£4 087,00**
en el escenario más exigente. Añadiría una revisión comercial de pedidos grandes confirmados, en lugar de
cambiar automáticamente a un método que ya falló en el único test separado.

La evaluación sigue siendo académica: los valores en libras son supuestos, no costos observados. Además,
solo se reservó una ventana de ocho semanas y los modelos no reciben información anticipada de campañas.


## Reto 2 — ¿Conviene añadir un colchón?

**Caso.** El área comercial propone planificar unidades adicionales cada semana para reducir faltantes. Almacén advierte que el excedente también cuesta: espacio, capital inmovilizado y posibles liquidaciones. Se debe decidir con evidencia si conviene añadir un colchón al pronóstico.

**Problema.** ¿Qué política de cantidad, sin colchón o con colchón de 0,5; 1 o 1,5 veces el MAE previo, debería usarse, y qué costo produjo en test frente a no usar colchón?

**Datos, supuestos y recursos**
- Modelo seleccionado en la parte guiada (`SELECCIONADO`) y sus pronósticos de validación y test.
- Colchón = ⌈multiplicador × MAE previo⌉; el MAE previo usa solo los 2 bloques de 8 semanas anteriores a cada origen (`mae_previo`, `colchon`, `comparar_politicas`).
- Costos base: £1,50 por faltante y £0,40 por exceso (supuestos académicos). Regla común de selección.

**Apartados**
- **A.** Definir pronóstico, cantidad planificada y políticas
- **B.** Comparar las políticas en validación
- **C.** Seleccionar el multiplicador y evaluarlo en test
- **D.** Recomendar sobre el colchón

**Evidencias que deben quedar en el notebook**
- Tabla del MAE previo y del colchón de cada política por fold.
- Costos por fold y resumen por política con excedentes, faltantes y folds en que el colchón mejoró, más el gráfico de excedentes y faltantes.
- Tabla de test de la política seleccionada frente a la opción sin colchón.
- Interpretaciones A–D.

### Reto 2 · Apartado A — Definir pronóstico, cantidad planificada y políticas
Explique la diferencia entre pronóstico y cantidad planificada. Calcule, para cada fold, el MAE previo (solo con semanas anteriores a su origen) y el colchón que tendría cada política (0; 0,5; 1 y 1,5 × MAE previo).

In [21]:
# MULTIPLICADORES = [0, 0.5, 1.0, 1.5] ya está definido en la parte guiada
# Para cada fold, el MAE previo usa solo los 2 bloques de 8 semanas anteriores a su origen
filas = []
for k, o in enumerate(origenes, start=1):
    mae_k = mae_previo(SELECCIONADO, train.iloc[:o], f"reto2 · fold {k}")
    fila = {"fold": k, "ultima_semana_calibracion": train.index[o - 1].date(), "inicio_validacion": train.index[o].date(),
            "mae_previo": mae_k}
    fila.update({f"colchón {m:g}×": colchon(m, mae_k) for m in MULTIPLICADORES})
    filas.append(fila)
r2_prep = pd.DataFrame(filas).set_index("fold")
display(r2_prep.round(1))

,ultima_semana_calibracion,inicio_validacion,mae_previo,colchón 0×,colchón 0.5×,colchón 1×,colchón 1.5×
fold,,,,,,,
1,2010-12-27,2011-01-03,1400.7,0,701,1401,2102
2,2011-02-21,2011-02-28,1533.2,0,767,1534,2300
3,2011-04-18,2011-04-25,1274.1,0,638,1275,1912
4,2011-06-13,2011-06-20,899.9,0,450,900,1350
5,2011-08-08,2011-08-15,821.3,0,411,822,1232


**Interpretación 2A · Qué representa el colchón:**

Distingo la **predicción de ventas** de la **cantidad que quiero tener disponible**. Para esta última puedo
agregar un margen que dependa del error pasado. Lo calculé antes de cada origen de validación, usando las
16 semanas previas repartidas en dos bloques: el MAE de calibración fue de aproximadamente **1 400,7;
1 533,2; 1 274,1; 899,9 y 821,3** unidades. Los primeros valores son altos porque reciben el efecto
posterior a la temporada con grandes pedidos. Esto demuestra que la política de inventario también puede
ser volátil si se basa en errores excepcionales.


### Reto 2 · Apartado B — Comparar las políticas en validación
Evalúe las cuatro políticas en los folds de validación con calibración temporal (`comparar_politicas`). Presente el costo por fold, el resumen por política (costo promedio, desviación, excedentes y faltantes), cuántos folds mejoró cada colchón frente a no usarlo y un gráfico de excedentes y faltantes.

In [22]:
r2_folds = comparar_politicas(etiqueta="reto2 · validación")
r2_resumen = resumir(r2_folds, "multiplicador", MULTIPLICADORES)

# Costo por fold de cada política y número de folds en que cada colchón costó menos que no usarlo (0×)
costo_fold = r2_folds.pivot(index="fold", columns="multiplicador", values="costo")
costo_fold.columns = [f"{m:g}×" for m in costo_fold.columns]
display(costo_fold.round(1))
r2_resumen["folds_mejor_que_0x"] = [np.nan if m == 0 else int((costo_fold[f"{m:g}×"] < costo_fold["0×"]).sum())
                                    for m in MULTIPLICADORES]
display(r2_resumen[["costo_prom", "costo_sd", "MAE_prom", "excedentes", "faltantes", "folds_mejor_que_0x"]].round(2))

fig = go.Figure([go.Bar(x=[f"{m:g}×" for m in MULTIPLICADORES], y=r2_resumen["excedentes"], name="Unidades excedentes"),
                 go.Bar(x=[f"{m:g}×" for m in MULTIPLICADORES], y=r2_resumen["faltantes"], name="Unidades faltantes")])
fig.update_layout(barmode="group", title="Reto 2 · Excedentes y faltantes en validación por política (5 folds)",
                  xaxis_title="Multiplicador del MAE previo", yaxis_title="Unidades")
fig.show()

,0×,0.5×,1×,1.5×
fold,,,,
1,6567.0,7478.3,8546.0,10789.2
2,6713.4,5345.0,5555.5,7723.6
3,4286.6,4563.1,5664.8,7703.2
4,2534.0,3974.0,5414.0,6854.0
5,1519.5,2673.2,3988.4,5300.4


,costo_prom,costo_sd,MAE_prom,excedentes,faltantes,folds_mejor_que_0x
multiplicador,,,,,,
0.0,4324.10,2335.11,927.30,30925.0,6167.0,NaN
0.5,4806.72,1784.08,1334.35,50934.0,2440.0,1.0
1.0,5833.74,1661.61,1812.80,72363.0,149.0,1.0
1.5,7674.08,2000.88,2398.15,95926.0,0.0,0.0


**Interpretación 2B · Balance costo / nivel de servicio:**

El costo de validación aumenta con el multiplicador: **£4 324,10** para 0×, **£4 806,72** para 0,5×,
**£5 833,74** para 1× y **£7 674,08** para 1,5×. Es cierto que el número de unidades faltantes baja
de **6 167 a 0**, pero los sobrantes suben de **30 925 a 95 926** en el extremo de 1,5×.

Yo no agregaría inventario solo porque suena más seguro: cada unidad que sobra tiene costo. Para justificar
el colchón tendría que contar con costos empresariales reales, previsión de demanda y restricciones de entrega,
no únicamente con el deseo de eliminar cualquier faltante.


### Reto 2 · Apartado C — Seleccionar el multiplicador y evaluarlo en test
Seleccione el multiplicador con la regla común (sin mirar el test). Calcule el colchón del test con el MAE previo del entrenamiento y evalúe la política seleccionada frente a la opción sin colchón: costo, excedentes y faltantes. Compruebe si coincide con la política fijada en la sección 10.

In [23]:
r2_mult = seleccionar(r2_resumen)
# Colchón del test: MAE previo calculado solo con el entrenamiento (misma regla de 2 bloques)
mae_t = mae_previo(SELECCIONADO, train, "reto2 · test")
colchon_t = colchon(r2_mult, mae_t)
tabla_r2c = pd.DataFrame({
    f"{SELECCIONADO} sin colchón": evaluar(test, cantidades_test[SELECCIONADO]),
    f"Política seleccionada: {SELECCIONADO} + {r2_mult:g} × MAE previo (colchón {colchon_t})":
        evaluar(test, cantidades_test[SELECCIONADO] + colchon_t)}).T
display(tabla_r2c.round(2))
print(f"Multiplicador seleccionado en validación: {r2_mult:g} · MAE previo del test = {mae_t:.1f} · colchón = {colchon_t} unidades por semana")
print("¿Coincide con la política fijada en la sección 10?", "Sí" if r2_mult == MULT else "No")

,MAE,RMSE,WAPE,unidades_excedentes,unidades_faltantes,costo
ExpSmoothing sin colchón,776.62,1700.86,0.54,329.0,5884.0,8957.6
Política seleccionada: ExpSmoothing + 0 × MAE previo (colchón 0),776.62,1700.86,0.54,329.0,5884.0,8957.6


Multiplicador seleccionado en validación: 0 · MAE previo del test = 618.8 · colchón = 0 unidades por semana
¿Coincide con la política fijada en la sección 10? Sí


**Interpretación 2C:**
La regla común selecciona el multiplicador **0** (sin colchón): es el de menor costo promedio (£4 324,10) y el único candidato, porque el límite del 1 % es £4 367,34. El colchón del test es 0 unidades (MAE previo del entrenamiento = 618,8), así que la política seleccionada y la opción sin colchón son idénticas en el test: £8 957,60, con 329 excedentes y 5 884 faltantes. **Coincide** con la política fijada en la sección 10.

### Reto 2 · Apartado D — Recomendar sobre el colchón
Recomiende si corresponde añadir colchón y fundamente con la validación. Explique por qué la cantidad objetivo semanal no equivale automáticamente a una cantidad a comprar (inventario utilizable, pedidos que llegarán a tiempo y arrastre entre semanas).

**Decisión 2D:**
**No corresponde añadir colchón** con este modelo. En validación, no usar colchón fue lo más barato en promedio y ganó en 4 de los 5 folds; cada colchón probado aumentó el costo promedio entre £482,62 (0,5×) y £3 349,98 (1,5×). Retrospectivamente, en el test un colchón de 0,5× habría bajado el costo a £7 247,80, pero solo por el pedido extraordinario de 4 800 unidades; aun así habría seguido por encima de Naïve (£7 087,90). Es un resultado mirado después y no justifica cambiar la política.

La **cantidad objetivo semanal** no es una cantidad a comprar. Para pasar a una compra hay que restar el **inventario utilizable** (apto para la venta) y los **pedidos que llegarán a tiempo**, y considerar el **arrastre entre semanas**: lo que sobra una semana reduce lo que se necesita la siguiente. Por eso sumar los objetivos semanales sobreestima lo que realmente habría que pedir.

### Reto 2 · Ampliación opcional: ventana de calibración
Repita la comparación con 1, 2 y 3 bloques de calibración y observe si cambia el multiplicador elegido. Si muestra costos de test de las políticas descartadas, identifíquelos como retrospectivos: no se usan para cambiar la política.

In [24]:
# Opcional: para b en [1, 2, 3], use comparar_politicas(bloques=b, etiqueta=...) , resumir(...) y seleccionar(...)
filas = []
for b in [1, 2, 3]:
    res_b = resumir(comparar_politicas(bloques=b, etiqueta=f"reto2 · ventana {b}"), "multiplicador", MULTIPLICADORES)
    m_b = seleccionar(res_b)
    filas.append({"bloques_calibracion": b, "multiplicador_elegido": m_b,
                  **{f"costo_prom {m:g}×": res_b.loc[m, "costo_prom"] for m in MULTIPLICADORES}})
ventanas = pd.DataFrame(filas).set_index("bloques_calibracion")
display(ventanas.round(2))

# RETROSPECTIVO: costo en test de todas las políticas (2 bloques). Solo informativo; no cambia la política fijada.
retro = pd.DataFrame({f"{m:g}×": evaluar(test, cantidades_test[SELECCIONADO] + colchon(m, mae_t)) for m in MULTIPLICADORES}).T
retro["colchón"] = [colchon(m, mae_t) for m in MULTIPLICADORES]
print("Costos de test RETROSPECTIVOS (no se usan para cambiar la política):")
display(retro[["colchón", "unidades_excedentes", "unidades_faltantes", "costo"]].round(2))

,multiplicador_elegido,costo_prom 0×,costo_prom 0.5×,costo_prom 1×,costo_prom 1.5×
bloques_calibracion,,,,,
1,0.0,4324.1,4744.00,5704.00,7414.24
2,0.0,4324.1,4806.72,5833.74,7674.08
3,0.0,4324.1,4870.40,5927.90,7663.84


Costos de test RETROSPECTIVOS (no se usan para cambiar la política):


,colchón,unidades_excedentes,unidades_faltantes,costo
0×,0,329.0,5884.0,8957.6
0.5×,310,1387.0,4462.0,7247.8
1×,619,3550.0,4153.0,7649.5
1.5×,929,5720.0,3843.0,8052.5


**Comentario opcional 2:**
Con 1, 2 y 3 bloques de calibración se elige siempre el multiplicador 0: cambiar la ventana mueve un poco el costo de los colchones, pero en ningún caso alguno queda por debajo de no usar colchón. La decisión no depende de la ventana elegida. Los costos de test de las políticas descartadas (0,5× ≈ £7 247,80; 1× ≈ £7 649,50; 1,5× ≈ £8 052,50) son **retrospectivos**: se muestran solo como referencia y no cambian la política fijada.

## Reto 3 — ¿Funciona la misma solución para otro producto?

**Caso.** La empresa quiere extender el procedimiento a un segundo producto de alta rotación antes de aplicarlo a toda la cartera. La gerencia pregunta si el método que se eligió para el primer producto puede generalizarse.

**Problema.** ¿Qué modelo se selecciona para el segundo producto con las mismas reglas, cómo le fue en test frente a Naïve y qué haría la empresa con cada producto?

**Datos, supuestos y recursos**
- Segundo SKU de la lista de elegibles, definida solo con el entrenamiento (no se elige mirando el test).
- Mismas semanas completas, limpieza, backtesting, costos base y regla común de selección.
- Se comparan modelos **sin colchón** para mantener el alcance claro; la política de colchón se trabajó en el Reto 2.

**Apartados**
- **A.** Identificar el producto y describir su serie
- **B.** Backtesting y selección con la regla común
- **C.** Evaluar en test y comparar ambos productos
- **D.** Recomendación consolidada

**Evidencias que deben quedar en el notebook**
- Serie del segundo producto y tabla descriptiva del entrenamiento frente al primero, con su gráfico.
- Tabla de backtesting por modelo y seleccionado.
- Tabla comparativa de ambos productos con modelo, costos, ahorro o sobrecosto y resultado.
- Interpretaciones A–D.

### Reto 3 · Apartado A — Identificar el producto y describir su serie
Identifique el segundo SKU elegible, construya su serie con las mismas semanas completas y separe entrenamiento y test. Describa su comportamiento solo con el entrenamiento y compárelo con el primer producto (tabla y gráfico).

In [25]:
assert len(elegibles) >= 2, "Se necesita un segundo SKU elegible"
SKU2 = elegibles.loc[1, "StockCode"]
# Serie del segundo producto con la misma construcción de la sección 6 (mismas semanas completas y ceros explícitos)
desc2 = uk.loc[uk["StockCode"] == SKU2, "Description"].dropna().astype(str).mode().iloc[0]
serie2 = (uk[uk["StockCode"] == SKU2].groupby("week")["Quantity"].sum()
          .reindex(semanas, fill_value=0).astype(float))
serie2.name = "unidades"
train2, test2 = serie2.loc[semanas_train], serie2.loc[semanas_test]
print("Segundo SKU:", SKU2, "·", desc2)

def describir(y):
    """Estadísticos descriptivos de una serie de entrenamiento."""
    return {"unidades": y.sum(), "media": y.mean(), "mediana": y.median(), "desv_est": y.std(),
            "coef_variacion": y.std() / y.mean(), "maximo": y.max(), "semanas_sin_venta": int((y == 0).sum()),
            "media_2010": y[y.index.year == 2010].mean(), "media_2011": y[y.index.year == 2011].mean(),
            "media_ultimas_8": y.iloc[-8:].mean()}

r3_desc = pd.DataFrame({f"{SKU} · {desc}": describir(train), f"{SKU2} · {desc2}": describir(train2)}).T
display(r3_desc.round(2))

fig = go.Figure([go.Scatter(x=train.index, y=train.values, name=f"{SKU} · {desc}"),
                 go.Scatter(x=train2.index, y=train2.values, name=f"{SKU2} · {desc2}")])
fig.update_layout(title="Reto 3 · Ventas semanales registradas en el entrenamiento",
                  xaxis_title="Semana (lunes)", yaxis_title="Unidades")
fig.show()

Segundo SKU: 85123A · WHITE HANGING HEART T-LIGHT HOLDER


,unidades,media,mediana,desv_est,coef_variacion,maximo,semanas_sin_venta,media_2010,media_2011,media_ultimas_8
84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS,88273.0,919.51,584.5,1080.36,1.17,7063.0,3.0,1034.44,812.75,653.25
85123A · WHITE HANGING HEART T-LIGHT HOLDER,80903.0,842.74,718.5,582.80,0.69,3329.0,2.0,977.12,641.52,460.00


**Interpretación 3A:**
El segundo SKU elegible es **85123A · WHITE HANGING HEART T-LIGHT HOLDER**, elegido con la lista definida solo con el entrenamiento. Comparado con el primero (84077):
- Vende parecido en volumen (80 903 frente a 88 273 unidades), pero es mucho más **estable**: desviación de 582,80 frente a 1 080,36 y coeficiente de variación de 0,69 frente a 1,17.
- Su pico máximo es menor (3 329 frente a 7 063) y su mediana (718,5) está más cerca de la media (842,74), así que depende menos de pedidos extraordinarios.
- Tiene 2 semanas sin ventas (las de posible cierre) frente a 3.
- Su nivel cae más en 2011: la media pasa de 977,12 en 2010 a 641,52 en 2011 (−34 %), y en las últimas 8 semanas del entrenamiento promedia 460 unidades (653,25 en el primer producto).

Es una serie más regular, pero con un nivel decreciente.

### Reto 3 · Apartado B — Backtesting y selección con la regla común
Ejecute el mismo backtesting con Naïve, Naïve estacional y Exponential Smoothing (`backtest`), resuma los resultados y seleccione con la regla común. Presente la tabla e indique si la selección se decidió por la tolerancia del 1 %.

In [26]:
filas2, _ = backtest(train2, MODELOS, "reto3")
tabla2 = resumir(filas2, "modelo", MODELOS)
SELECCIONADO2 = seleccionar(tabla2)
# Marca qué modelos quedaron dentro de la tolerancia del 1 % (candidatos de la regla común)
tabla2["candidato_1pct"] = tabla2["costo_prom"] <= tabla2["costo_prom"].min() * (1 + TOLERANCIA) + 1e-9
display(tabla2.round(2))
print("Candidatos dentro del 1 %:", list(tabla2.index[tabla2["candidato_1pct"]]))
print("MODELO SELECCIONADO PARA EL SEGUNDO PRODUCTO (antes del test):", SELECCIONADO2)

,costo_prom,costo_sd,MAE_prom,MAE_sd,RMSE_prom,WAPE_prom,excedentes,faltantes,complejidad,candidato_1pct
modelo,,,,,,,,,,
Naive,4814.82,3483.34,670.62,656.73,867.66,0.97,14694.0,12131.0,0,False
SeasonalNaive,3306.84,2147.86,459.32,290.28,665.50,0.67,10023.0,8350.0,1,False
ExpSmoothing,2591.80,1907.11,475.12,266.42,569.80,0.71,14135.0,4870.0,2,True


Candidatos dentro del 1 %: ['ExpSmoothing']
MODELO SELECCIONADO PARA EL SEGUNDO PRODUCTO (antes del test): ExpSmoothing


**Interpretación 3B:**
Con la misma regla común se selecciona **Exponential Smoothing**: costo promedio de validación de £2 591,66 (desviación £1 906,87), frente a £3 306,84 de Naïve estacional y £4 814,82 de Naïve. Fue el único candidato (límite del 1 % ≈ £2 617,58), así que la selección no se decidió por la tolerancia.

Igual que en el primer producto, el ranking por MAE no coincide con el de costo: Naïve estacional tiene un MAE ligeramente menor (459,32 frente a 474,88), pero más faltantes (8 350 frente a 4 873), y eso lo hace más caro.

### Reto 3 · Apartado C — Evaluar en test y comparar ambos productos
Evalúe en test el modelo seleccionado y Naïve para el segundo producto. Construya una tabla comparativa de ambos productos (sin colchón) con modelo seleccionado, costo de validación, costo de test del seleccionado y de Naïve, ahorro o sobrecosto, folds en que superó a Naïve y resultado de la evaluación, aplicando la regla de adopción de la sección 13.

In [27]:
# Segundo producto: pronóstico de test con el modelo seleccionado y con Naïve, entrenando con train2
q2_sel = pronosticar(SELECCIONADO2, train2, TEST_H, "reto3 · test")
q2_naive = pronosticar("Naive", train2, TEST_H, "reto3 · test")
ev2_sel, ev2_naive = evaluar(test2, q2_sel), evaluar(test2, q2_naive)
bt2 = pd.DataFrame(filas2)

def resultado_adopcion(costo_sel, costo_nv, folds, modelo):
    """Regla de adopción de la sección 13."""
    if modelo == "Naive":
        return "El seleccionado es el propio Naïve: se mantiene Naïve como referencia"
    if costo_sel < costo_nv and folds > len(origenes) / 2:
        return "Adoptar como piloto controlado frente a Naïve"
    if costo_sel < costo_nv:
        return "Mantener en evaluación"
    return "No adoptar como sustituto de Naïve; Naïve se mantiene como referencia"

def folds_vs_naive(bt_df, modelo):
    """Folds de validación en que el modelo costó menos que Naïve."""
    c = bt_df.pivot(index="fold", columns="modelo", values="costo")
    return int((c[modelo] < c["Naive"]).sum())

# Primer producto (sin colchón): objetos de la parte guiada
c1_sel = res_test.loc[f"{SELECCIONADO} (sin colchón)", "costo"]
c1_nv = res_test.loc["Naive (sin colchón)", "costo"]
f1 = folds_vs_naive(bt, SELECCIONADO)
# Segundo producto
f2 = folds_vs_naive(bt2, SELECCIONADO2)

comparacion = pd.DataFrame([
    {"producto": f"{SKU} · {desc}", "modelo_seleccionado": SELECCIONADO,
     "costo_validacion_prom": resumen_val.loc[SELECCIONADO, "costo_prom"],
     "costo_test_seleccionado": c1_sel, "costo_test_Naive": c1_nv, "ahorro_vs_Naive": c1_nv - c1_sel,
     "folds_supero_Naive": f"{f1} de {len(origenes)}", "resultado": resultado_adopcion(c1_sel, c1_nv, f1, SELECCIONADO)},
    {"producto": f"{SKU2} · {desc2}", "modelo_seleccionado": SELECCIONADO2,
     "costo_validacion_prom": tabla2.loc[SELECCIONADO2, "costo_prom"],
     "costo_test_seleccionado": ev2_sel["costo"], "costo_test_Naive": ev2_naive["costo"],
     "ahorro_vs_Naive": ev2_naive["costo"] - ev2_sel["costo"],
     "folds_supero_Naive": f"{f2} de {len(origenes)}", "resultado": resultado_adopcion(ev2_sel["costo"], ev2_naive["costo"], f2, SELECCIONADO2)},
]).set_index("producto")
display(comparacion.round(2))
print(f"Segundo producto en test · {SELECCIONADO2}: excedentes {ev2_sel['unidades_excedentes']:.0f}, faltantes {ev2_sel['unidades_faltantes']:.0f} · "
      f"Naïve: excedentes {ev2_naive['unidades_excedentes']:.0f}, faltantes {ev2_naive['unidades_faltantes']:.0f}")

,modelo_seleccionado,costo_validacion_prom,costo_test_seleccionado,costo_test_Naive,ahorro_vs_Naive,folds_supero_Naive,resultado
producto,,,,,,,
84077 · WORLD WAR 2 GLIDERS ASSTD DESIGNS,ExpSmoothing,4324.1,8957.6,7087.9,-1869.7,4 de 5,No adoptar como sustituto de Naïve; Naïve se m...
85123A · WHITE HANGING HEART T-LIGHT HOLDER,ExpSmoothing,2591.8,3432.0,6640.7,3208.7,5 de 5,Adoptar como piloto controlado frente a Naïve


Segundo producto en test · ExpSmoothing: excedentes 810, faltantes 2072 · Naïve: excedentes 8, faltantes 4425


**Interpretación 3C:**
Los resultados son opuestos:
- **Segundo producto (85123A):** Exponential Smoothing costó £3 376,40 en el test frente a £6 640,70 de Naïve, un **ahorro simulado de £3 264,30 (49,2 %)**, y superó a Naïve en los 5 folds de validación. Con la regla de la sección 13, el resultado es **adoptar como piloto controlado**. Naïve repitió una semana baja y dejó 4 425 unidades faltantes; Exponential Smoothing dejó 2 036 faltantes y 806 excedentes.
- **Primer producto (84077):** el mismo método costó £8 957,60 frente a £7 087,90 de Naïve (sobrecosto de £1 869,70), aunque había ganado 4 de 5 folds; el resultado es **no adoptar**.

El mismo procedimiento funcionó en la serie más estable y falló en la que depende de pedidos extraordinarios.

### Reto 3 · Apartado D — Recomendación consolidada
Prepare una recomendación para la empresa: qué haría con cada producto (adoptar, mantener en evaluación o no adoptar), qué evidencia la respalda y por qué dos productos no permiten generalizar el método a toda la cartera.

**Decisión 3D:**
- **Producto 84077 (WORLD WAR 2 GLIDERS):** **no adoptar** Exponential Smoothing; Naïve sigue como referencia. Ganó en validación, pero costó 26,4 % más en el test, en una serie muy irregular y con pedidos grandes difíciles de anticipar.
- **Producto 85123A (WHITE HANGING HEART T-LIGHT HOLDER):** **piloto controlado** con Exponential Smoothing. Ganó a Naïve en los 5 folds y en el test (ahorro simulado de £3 264,30). El piloto debe comprobar en semanas nuevas, con la misma regla, que la reducción de costo se mantiene antes de decidir compras con él.

**Por qué no se puede generalizar:** con dos productos el método dio dos decisiones distintas. Son solo 2 de los 389 SKUs elegibles, ambos de los más vendidos (no representan a los de baja rotación), y cada uno tiene un solo test de 8 semanas. Para extender el procedimiento a la cartera habría que evaluar producto por producto, o por grupos con comportamiento parecido, y en más ventanas de prueba.

### Revisión de advertencias y verificaciones automáticas

Se resumen las advertencias de **todos** los ajustes de Exponential Smoothing (backtesting, calibración, test, reentrenamiento final y retos) y se comprueban las condiciones de validez del flujo.

In [28]:
aj = pd.DataFrame(registro_ajustes)
display(aj.groupby("etapa").agg(ajustes=("ajuste", "count"), con_advertencias=("advertencias", lambda s: int((s > 0).sum())),
                                tipos=("tipos", lambda s: "; ".join(sorted({t for x in s for t in x.split("; ") if t}))))
          .sort_values("ajustes", ascending=False))
mensajes = sorted({m for x in aj["mensajes"] for m in x.split(" | ") if m})
print("Mensajes distintos:", mensajes if mensajes else "ninguno")
print("Especificaciones usadas:", aj["especificacion"].value_counts().to_dict())

verificaciones = {
    "Ningún fold invade el test": all(train.index[o + BT_H - 1] < semanas_test[0] for o in origenes),
    "Colchón de cada fold calibrado solo con semanas anteriores a su origen": bool((pol_folds["ultima_semana_calibracion"] < pol_folds["inicio_validacion"]).all()),
    "Colchón del test calculado solo con el entrenamiento": len(train) == len(semanas_train) and train.index[-1] < semanas_test[0],
    "La tabla final usa el multiplicador fijado": COLCHON_FINAL == colchon(MULT, MAE_FINAL),
    "El costo de la política en test se recalcula con su colchón": np.isclose(costo_pol, evaluar(test, q_politica)["costo"]),
}
for k, v in verificaciones.items():
    print(("OK   " if v else "FALLA"), k)

,ajustes,con_advertencias,tipos
etapa,,,
reto2,52,0,
calibración,10,0,
reto3,6,0,
backtesting,5,0,
calibración test,2,0,
calibración final,2,0,
final,1,0,
test,1,0,


Mensajes distintos: ninguno
Especificaciones usadas: {'nivel + tendencia + estacionalidad aditivas (m = 4)': 79}
OK    Ningún fold invade el test
OK    Colchón de cada fold calibrado solo con semanas anteriores a su origen
OK    Colchón del test calculado solo con el entrenamiento
OK    La tabla final usa el multiplicador fijado
OK    El costo de la política en test se recalcula con su colchón


### Retroalimentación final

**Pregunta 10.** Indique dos razones por las que el modelo podría fallar en producción aun cuando el backtesting haya sido bueno.

**Respuesta desde una perspectiva de operación y abastecimiento:**
1. **Pedidos inesperados y campañas que no aparecen en la serie:** las ventas dependen mucho de pedidos grandes de pocos clientes mayoristas (en el test, un solo pedido fue el 90 % de la semana pico). Un modelo univariante solo mira las ventas pasadas, así que una campaña, un cliente nuevo o uno que deja de comprar cambian la serie sin aviso.
2. **Condiciones operativas distintas a las del histórico:** el backtesting cubrió de enero a octubre de 2011 y el entrenamiento solo tiene un otoño. Si la temporada navideña, los cierres de fin de año o el nivel de ventas cambian, el modelo proyecta un patrón que ya no aplica. A eso se suma que los ceros pueden ser faltas de inventario y no demanda nula, y que los costos reales pueden ser distintos a los supuestos con los que se eligió el modelo.


# 15. Ticket de salida · Mi decisión

En el análisis de referencia el método elegido mediante validación fue **Exponential Smoothing** con política
**0 × MAE**, es decir, sin unidades adicionales. Su costo medio de validación fue **£4 324,10**.

Cuando lo confronté con el periodo final, el costo fue **£8 957,60** versus **£7 087,90** del Naïve. Esto
representa un **sobrecosto de £1 869,70 (26,4 %)**: mi propuesta es **no implementarlo como reemplazo** hasta
repetir la evaluación en fechas nuevas. Aunque el objetivo académico proyecta **9 291 unidades** para ocho
semanas, todavía faltaría estimar las compras netas considerando inventario, entregas en tránsito y tiempos de
reposición. Mi prioridad es evitar que una buena métrica se convierta en una mala orden de compra.

> Confirmar que los números anteriores coincidan con la ejecución actual antes de usar esta respuesta final.


# 16. Mis conclusiones técnicas y comerciales

1. **No elegiría un método solo porque gana en la validación.** En el experimento de referencia, Exponential Smoothing tuvo mejor costo medio en los folds (£4 324,10), pero perdió en las ocho semanas de test y generó un sobrecosto del 26,4 % frente al Naïve. Antes de cambiar un proceso, pediría más evidencia temporal.
2. **La métrica adecuada depende del negocio.** Un faltante y un excedente tienen efectos diferentes. El MAE ayuda a expresar el error en unidades, pero no incorpora cuánto cuesta perder una venta. Por eso compararía costo, sesgo y nivel de atención conjuntamente; en una empresa de maquinaria también separaría repuestos críticos de artículos comunes.
3. **No trasladaría un resultado directamente a toda la cartera.** Para el SKU 85123A, el suavizamiento exponencial logró un ahorro simulado del 49,2 %, mientras que para 84077 no fue conveniente. Mi siguiente paso sería clasificar productos por estabilidad, criticidad y frecuencia de pedidos antes de escalar el modelo.

**Advertencia metodológica:** estas cifras corresponden al escenario académico del notebook de referencia y deben
coincidir con la ejecución reproducida. El archivo de UCI no ofrece el stock real, tiempos de compra ni pérdidas
comerciales observadas; por eso los importes utilizados son supuestos de evaluación, no utilidades contables.


# 17. Ampliación propia · Tablero para una decisión operativa

Quiero comprobar cómo se traducen las predicciones en indicadores que puedan entender comercial, almacén y
servicio técnico. **Esta sección es retrospectiva:** solo lee los pronósticos y el test que ya fueron calculados;
no altera el modelo seleccionado, la política ni la evaluación de las secciones anteriores.

En una venta de maquinaria, una pieza de reposición crítica puede generar un costo por falta muy diferente al
de un artículo de regalo. Aquí conservo los precios y productos de UCI únicamente como ejercicio estadístico.


In [29]:
# Panel de riesgos del test: comparación operativa, sin volver a seleccionar el modelo.
operativo = pd.DataFrame({
    "semana": pd.to_datetime(test.index),
    "venta_observada": test.to_numpy(dtype=float),
    "objetivo_modelo": np.asarray(q_politica, dtype=float),
    "objetivo_naive": np.asarray(cantidades_test["Naive"], dtype=float),
})

for prefijo, objetivo in [("modelo", "objetivo_modelo"), ("naive", "objetivo_naive")]:
    operativo[f"faltante_{prefijo}"] = np.maximum(operativo["venta_observada"] - operativo[objetivo], 0)
    operativo[f"excedente_{prefijo}"] = np.maximum(operativo[objetivo] - operativo["venta_observada"], 0)
    operativo[f"costo_{prefijo}"] = (operativo[f"faltante_{prefijo}"] * SHORT_COST
                                      + operativo[f"excedente_{prefijo}"] * OVER_COST)

def indicador_servicio(demanda, cantidad):
    """Porcentaje idealizado de demanda cubierta, con semanas independientes (sin stock arrastrado)."""
    d = np.asarray(demanda, dtype=float)
    q = np.asarray(cantidad, dtype=float)
    return np.minimum(d, q).sum() / d.sum() if d.sum() > 0 else np.nan

kpis_operativos = pd.DataFrame([
    {"alternativa": "Política seleccionada", "atencion_demanda": indicador_servicio(test, q_politica),
     "sesgo_medio_unidades": float(np.mean(np.asarray(q_politica) - test.to_numpy())),
     "faltantes": operativo["faltante_modelo"].sum(), "excedentes": operativo["excedente_modelo"].sum(),
     "costo_simulado_GBP": operativo["costo_modelo"].sum()},
    {"alternativa": "Naïve", "atencion_demanda": indicador_servicio(test, cantidades_test["Naive"]),
     "sesgo_medio_unidades": float(np.mean(np.asarray(cantidades_test["Naive"]) - test.to_numpy())),
     "faltantes": operativo["faltante_naive"].sum(), "excedentes": operativo["excedente_naive"].sum(),
     "costo_simulado_GBP": operativo["costo_naive"].sum()},
]).set_index("alternativa")

print("Indicadores del test (son diagnósticos, no criterios para recalibrar):")
display(kpis_operativos.round(3))
assert np.isclose(kpis_operativos.loc["Política seleccionada", "costo_simulado_GBP"], costo_pol)
assert np.isclose(kpis_operativos.loc["Naïve", "costo_simulado_GBP"], costo_naive)


Indicadores del test (son diagnósticos, no criterios para recalibrar):


,atencion_demanda,sesgo_medio_unidades,faltantes,excedentes,costo_simulado_GBP
alternativa,,,,,
Política seleccionada,0.485,-694.375,5884.0,329.0,8957.6
Naïve,0.609,-438.500,4469.0,961.0,7087.9


In [30]:
# Gráfico interactivo para localizar semanas con riesgo de subabastecimiento.
fig_ops = go.Figure()
for columna, etiqueta, modo in [
    ("venta_observada", "Venta registrada", "lines+markers"),
    ("objetivo_modelo", "Política seleccionada", "lines+markers"),
    ("objetivo_naive", "Baseline Naïve", "lines+markers"),
]:
    fig_ops.add_trace(go.Scatter(x=operativo["semana"], y=operativo[columna],
                                 mode=modo, name=etiqueta))
fig_ops.update_layout(title=f"Panel operativo | SKU {SKU} | Validación externa de 8 semanas",
                      xaxis_title="Semana", yaxis_title="Unidades", hovermode="x unified",
                      legend_title="Serie")
fig_ops.show()

vista_costos = operativo[["semana", "costo_modelo", "costo_naive"]].melt(
    id_vars="semana", var_name="alternativa", value_name="costo_semanal_GBP")
fig_costos = px.bar(vista_costos, x="semana", y="costo_semanal_GBP", color="alternativa",
                    barmode="group", title="Costo por semana: modelo seleccionado frente a Naïve")
fig_costos.update_layout(xaxis_title="Semana", yaxis_title="Costo simulado (£)")
fig_costos.show()


## 17.1. De cantidad objetivo a compra neta: tres escenarios de inventario

**Problema añadido por mí:** supongamos que el área de almacén ya dispone de unidades utilizables al comenzar
el horizonte. Si compra de nuevo todo lo que señala el pronóstico, puede duplicar inventario. Por ello simulo
la conversión de objetivos semanales a compras **sin pretender que estos datos sean inventario real de UCI**.

Para que el ejemplo se pueda repetir, empleo stocks iniciales hipotéticos de 0, 500 y 1 500 unidades, entrega
inmediata, sin pedidos en tránsito y consumo semanal igual al **pronóstico académico**, no a la venta futura
observada. En una empresa real tendría que agregar plazos, lotes mínimos, sucursales y reservas de clientes.


In [31]:
# Simulación didáctica sobre las 8 semanas FUTURAS; no se usa el test para calcular compras.
def plan_compras_netas(pronostico, objetivo, stock_inicial=0):
    pronostico = np.maximum(np.asarray(pronostico, dtype=int), 0)
    objetivo = np.maximum(np.asarray(objetivo, dtype=int), 0)
    if len(pronostico) != len(objetivo):
        raise ValueError("Pronóstico y objetivo requieren igual cantidad de semanas")
    if stock_inicial < 0:
        raise ValueError("El stock inicial no puede ser negativo")
    stock = int(stock_inicial)
    registros = []
    for num, (demanda_prevista, meta) in enumerate(zip(pronostico, objetivo), start=1):
        antes = stock
        compra = max(int(meta) - antes, 0)
        disponible = antes + compra   # solo bajo el supuesto hipotético de entrega inmediata
        stock = max(disponible - int(demanda_prevista), 0)
        registros.append({"semana_n": num, "stock_antes": antes, "objetivo": int(meta),
                          "compras_netas": compra, "consumo_previsto": int(demanda_prevista),
                          "stock_estimado_despues": stock})
    return pd.DataFrame(registros)

objetivos_futuros = (pron_final + COLCHON_FINAL).astype(int)
escenarios_stock = []
for inventario_hipotetico in (0, 500, 1500):
    simulacion = plan_compras_netas(pron_final, objetivos_futuros, inventario_hipotetico)
    escenarios_stock.append({
        "stock_inicial_sup": inventario_hipotetico,
        "suma_objetivos": int(objetivos_futuros.sum()),
        "suma_compras_netas_sup": int(simulacion["compras_netas"].sum()),
        "stock_final_proyectado": int(simulacion["stock_estimado_despues"].iloc[-1]),
        "diferencia_vs_suma_objetivos": int(objetivos_futuros.sum()-simulacion["compras_netas"].sum()),
    })
resultado_stock = pd.DataFrame(escenarios_stock)
display(resultado_stock)
print("Solo una simulación académica. La compra real también depende de tiempo de entrega, reservas y stock en tránsito.")


,stock_inicial_sup,suma_objetivos,suma_compras_netas_sup,stock_final_proyectado,diferencia_vs_suma_objetivos
0,0,9291,9291,0,0
1,500,9291,8791,0,500
2,1500,9291,7791,0,1500


Solo una simulación académica. La compra real también depende de tiempo de entrega, reservas y stock en tránsito.


### 17.2. Interpretación de mi ampliación

En mi opinión, el pronóstico es útil para planificar, pero no debe llegar directamente al área de compras.
El **nivel de atención simulado** muestra cuánto volumen se habría podido cubrir si cada semana dispusiera
exactamente de la cantidad objetivo, mientras que el **sesgo** evidencia si el modelo tiende a sobredimensionar
o quedarse corto. Ambos complementan el costo asimétrico, aunque no reemplazan una simulación con inventario
real y entregas.

Al cambiar el inventario inicial supuesto, cambia la suma de compras netas sin modificar el pronóstico.
Este es el punto que quiero resaltar desde la operación: **misma predicción ≠ misma decisión de compra**.
Para aplicar el método a repuestos de máquinas industriales, primero estimaría costos reales de inmovilización,
paradas por falta de piezas, plazo de reposición y criticidad por sucursal. Sin esa información, no recomendaría
automatizar órdenes de compra con esta salida.


# 18. Verificación previa a la entrega

- [x] Código que carga Online Retail II desde fuentes identificadas y documenta la limpieza.
- [x] Partición temporal; selección de productos sin mirar el test.
- [x] Tres pronósticos, validación con varios orígenes y política de costo asimétrico.
- [x] Tres retos de aplicación (escenarios de costo, colchón y segundo producto).
- [x] Extensión propia: indicadores operativos, visualización y cálculo de compras netas simuladas.
- [ ] **Ejecutar todas las celdas en mi cuenta y contrastar las cifras escritas con los resultados.**
- [ ] Completar mi sección académica.
- [ ] Guardar y compartir mi copia de Colab con permiso de lectura.
- [ ] Publicar el `.ipynb` en `seman 10/` y copiar el enlace real al archivo.

## Fuentes y atribución

- Chen, D. (2012). *Online Retail II* [Conjunto de datos]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3.ª ed.). OTexts.
- Sharda, R., Delen, D., & Turban, E. *Business Intelligence, Analytics, Data Science and AI*. Pearson.
- Notebook de laboratorio recibido como material de referencia. Adaptación, personalización y extensión realizadas con apoyo de IA; los resultados deben verificarse mediante ejecución propia.
